# BBUR burned-area Forecaster (lead-1) - Maritime West


Install the required packages.


In [ ]:
import sys, subprocess
def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
for pkg in ['catboost', 'xgboost', 'lightgbm', 'scikit-learn']:
    try: __import__(pkg.replace('-','_'))
    except ImportError: pip_install(pkg)
print('Dependencies OK')


Load the training dataset, GFED4s burned-area targets and optional GRACE data.


In [ ]:
import os, gc, subprocess, warnings
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.isotonic import IsotonicRegression
from catboost import CatBoostRegressor, CatBoostClassifier
import xgboost as xgb
import lightgbm as lgb
warnings.filterwarnings('ignore')

# GPU availability check -- CatBoost/XGBoost switch to CUDA automatically below if True.
# The per-pixel training loop deliberately forces CB/XGB/LightGBM to CPU regardless of
# this flag (GPU launch overhead dominates for the thousands of tiny per-pixel fits);
# GPU is only used for the large pooled hyperparameter-tuning cell.
try:
    GPU_AVAILABLE = subprocess.run(['nvidia-smi'], capture_output=True, text=True).returncode == 0
except FileNotFoundError:
    GPU_AVAILABLE = False
print(f'GPU available: {GPU_AVAILABLE}' + (' -- CatBoost/XGBoost (tuning) will use CUDA; per-pixel loop stays CPU regardless' if GPU_AVAILABLE else ' -- running on CPU'))

# Dataset paths -- 0.25deg, merged features (era5_deep_layers + era5_precip are now
# baked into this single file by process_all_kaggle_data_025.py; no longer separate .npz)
DATASET_PATH = '/kaggle/input/datasets/mahadiqbal061/bbur-enhanced/bbur_enhanced_training_ready_seasia_025.npz'
if not os.path.exists(DATASET_PATH):
    DATASET_PATH = os.path.join('..', 'data', 'BBUR_enhanced_kaggle_025',
                                'bbur_enhanced_training_ready_seasia_025.npz')
_BASE = os.path.dirname(DATASET_PATH)

arr           = np.load(DATASET_PATH, allow_pickle=True)
x_flat        = arr['x_flat'].astype(np.float32, copy=False)
y_flat        = arr['y_flat'].astype(np.float32, copy=False)
time_idx      = arr['time_idx'].astype(np.int32, copy=False)
lat_flat      = arr['lat_flat'].astype(np.float32, copy=False)
lon_flat      = arr['lon_flat'].astype(np.float32, copy=False)
feature_names = arr['feature_names'].astype(str).tolist()
years_pm      = arr['years_pm'].astype(np.int32, copy=False)
lat           = arr['lat'].astype(np.float32, copy=False)
lon           = arr['lon'].astype(np.float32, copy=False)
del arr; gc.collect()   # release the decompressed NpzFile buffers once arrays are pulled out

print(f'Grid   : {len(lat)} lat x {len(lon)} lon = {len(lat)*len(lon):,} pixels  (0.25deg)')
print(f'x_flat : {x_flat.shape}  ({x_flat.nbytes/1e9:.2f} GB in memory)')

# GFED4s TRUE BA target injection
GFED_PATH = os.path.join(_BASE, 'gfed4s_targets_seasia_025.npz')
if not os.path.exists(GFED_PATH):
    GFED_PATH = '/kaggle/input/datasets/mahadiqbal061/bbur-enhanced-025/gfed4s_targets_seasia_025.npz'
if not os.path.exists(GFED_PATH):
    GFED_PATH = os.path.join('..', 'data', 'GFED_enhanced_kaggle_025', 'gfed4s_targets_seasia_025.npz')

if os.path.exists(GFED_PATH):
    _gfed    = np.load(GFED_PATH, allow_pickle=True)
    _gfed_ba = _gfed['y_total_flat'].astype(np.float32, copy=False)
    _ba_col  = next(i for i, f in enumerate(feature_names) if 'burned_area' in f.lower())
    assert _gfed_ba.shape[0] == x_flat.shape[0], (
        f'GFED shape mismatch: {_gfed_ba.shape[0]} vs x_flat {x_flat.shape[0]}')
    x_flat[:, _ba_col] = _gfed_ba
    print(f'[GFED] Overwrote burned_area_pred with GFED4s y_total_flat '
          f'(col {_ba_col}, max={_gfed_ba.max():.2f} km2)')
    del _gfed, _gfed_ba; gc.collect()
else:
    print('[GFED] WARNING: gfed4s_targets_seasia_025.npz NOT FOUND -- '
          'using original burned_area_pred (model prediction, not ground truth)')

# NOTE: era5_deep_layers_seasia.npz / era5_precip_seasia.npz merge loop removed --
# both are now merged upstream into bbur_enhanced_training_ready_seasia_025.npz directly.

# Merge GRACE TWS data (gap-filled via forward-fill then back-fill; z-scored on 2003-2011 pre-test window)
import xarray as xr
if ' _BASE' not in dir() and '_BASE' not in dir():
    _BASE = os.path.dirname(DATASET_PATH)
GRACE_PATH = os.path.join(_BASE, 'GRCTellus.JPL.200204_202603.GLO.RL06.3M.MSCNv04CRI.nc')
if os.path.exists(GRACE_PATH):
    print('Loading GRACE TWS data (ffill/bfill gap-filling, pre-test 2003-2011 standardization)...')
    ds_grace   = xr.open_dataset(GRACE_PATH)
    lwe        = ds_grace['lwe_thickness']
    scale      = ds_grace['scale_factor']
    tws_scaled = lwe * scale.fillna(1.0)
    # Manuscript Sec 2.2(iii): gap-fill missing months by temporal forward-filling followed by back-filling
    tws_raw    = tws_scaled.resample(time='MS').mean().ffill(dim='time').bfill(dim='time')
    tws_lag1_raw  = tws_raw.shift(time=1).bfill(dim='time')
    tws_lag3_raw  = tws_raw.shift(time=3).bfill(dim='time')
    tws_lag6_raw  = tws_raw.shift(time=6).bfill(dim='time')
    tws_lag12_raw = tws_raw.shift(time=12).bfill(dim='time')
    tws_trend_6mo_raw      = tws_raw - tws_lag6_raw
    tws_trend_6mo_lag1_raw = tws_lag1_raw - tws_raw.shift(time=7).bfill(dim='time')

    n_times_g  = len(years_pm)
    time_start = pd.to_datetime('2003-01-01')
    time_end   = time_start + pd.DateOffset(months=n_times_g - 1)
    tgt_lon    = xr.DataArray(lon, dims='lon')
    tgt_lat    = xr.DataArray(lat, dims='lat')

    def _interp_grace(ts):
        return ts.sel(time=slice(time_start, time_end)).interp(
            lat=tgt_lat, lon=tgt_lon, method='nearest').values

    # Pre-test training period (2003-2011 = first 108 months) used exclusively for z-score stats
    _PRETEST_T = min(108, n_times_g)
    def _zscore_px_pretest(flat_arr):
        arr2d = flat_arr.reshape(n_times_g, -1)
        mu = arr2d[:_PRETEST_T].mean(axis=0, keepdims=True)
        si = arr2d[:_PRETEST_T].std(axis=0, keepdims=True) + 1e-8
        return ((arr2d - mu) / si).reshape(-1).astype(np.float32)

    _g_names = [
        'grace_tws_lag0', 'grace_tws_lag1', 'grace_tws_lag3',
        'grace_tws_lag6', 'grace_tws_lag12',
        'grace_tws_trend_6mo', 'grace_tws_trend_6mo_lag1',
    ]
    _g_raws = [
        tws_raw, tws_lag1_raw, tws_lag3_raw,
        tws_lag6_raw, tws_lag12_raw,
        tws_trend_6mo_raw, tws_trend_6mo_lag1_raw,
    ]
    _g_cols, _g_added = [], []
    for _gn, _gr in zip(_g_names, _g_raws):
        if _gn not in feature_names:
            _g_cols.append(_zscore_px_pretest(np.nan_to_num(_interp_grace(_gr).reshape(-1), nan=0.0)))
            _g_added.append(_gn)
    if _g_cols:
        g_feats       = np.stack(_g_cols, axis=1)
        x_flat        = np.concatenate([x_flat, g_feats], axis=1)
        feature_names = feature_names + _g_added
        print(f'GRACE TWS merged: +{len(_g_added)} features ({_g_added}) -> {x_flat.shape[1]} total')
        del g_feats
    del ds_grace, lwe, scale, tws_scaled; gc.collect()
else:
    print('GRACE TWS NetCDF not found -- continuing without external GRACE file')

n_times  = len(years_pm)
n_pixels = len(lat) * len(lon)

# ==============================================================================
# DEEP LAG FEATURE CONSTRUCTION (6-month and 12-month Precursors per Sec 2.2)
# ==============================================================================
print('\nConstructing explicit deep lag features (6mo and 12mo lags)...')
lag_targets = ['era5_z_500hPa_lag0', 'oni_lag0', 'era5_tp_lag0', 'era5_vpd', 'burned_area_pred']
shifts = [6, 12]

N_MONTHS = len(years_pm)
N_PIXELS = len(lat) * len(lon)

new_feats_list = []
new_names_list = []

for base_feat in lag_targets:
    if base_feat in feature_names:
        base_idx = feature_names.index(base_feat)
        base_2d  = x_flat[:, base_idx].reshape(N_MONTHS, N_PIXELS)
        for shift in shifts:
            new_name = base_feat.replace('_lag0', '') + f'_lag{shift}'
            if new_name in feature_names or new_name in new_names_list:
                continue
            shifted_2d = np.roll(base_2d, shift=shift, axis=0)
            shifted_2d[:shift, :] = 0.0
            new_feats_list.append(shifted_2d.reshape(-1, 1).astype(np.float32))
            new_names_list.append(new_name)

# Fallback for era5_swvl3_deficit_6mo if .npz predates 3-layer soil update
if 'era5_swvl3_deficit_6mo' not in feature_names and 'era5_swvl3_deficit_6mo' not in new_names_list:
    _sw_src = 'era5_swvl3_lag0' if 'era5_swvl3_lag0' in feature_names else (
              'era5_swvl1_lag0' if 'era5_swvl1_lag0' in feature_names else None)
    if _sw_src is not None:
        _sw_2d = np.nan_to_num(x_flat[:, feature_names.index(_sw_src)].reshape(N_MONTHS, N_PIXELS), nan=0.0)
        _sw_anom = np.zeros_like(_sw_2d)
        _pre_t = min(108, N_MONTHS)
        for _m in range(12):
            _m_pre = np.arange(_m, _pre_t, 12)
            _mu = np.nanmean(_sw_2d[_m_pre], axis=0, keepdims=True)
            _sd = np.nanstd(_sw_2d[_m_pre], axis=0, keepdims=True) + 1e-8
            _m_all = np.arange(_m, N_MONTHS, 12)
            _sw_anom[_m_all] = (_sw_2d[_m_all] - _mu) / _sd
        _sw_def6 = np.zeros_like(_sw_anom)
        for _t in range(N_MONTHS):
            _s0 = max(0, _t - 5)
            _sw_def6[_t] = np.sum(np.minimum(0.0, _sw_anom[_s0:_t + 1]), axis=0)
        new_feats_list.append(_sw_def6.reshape(-1, 1).astype(np.float32))
        new_names_list.append('era5_swvl3_deficit_6mo')
        print(f"Computed fallback 'era5_swvl3_deficit_6mo' from '{_sw_src}' (pre-test 2003-2011 climatology)")

# Build 1-month-antecedent equivalents for rolling/cumulative .npz features so lead-1 forecasters
# have lag-1-safe versions of every rolling dryness/vegetation stress indicator (Sec 2.3.1)
_ROLLING_NPZ_COLS = [
    'era5_swvl3_deficit_6mo',
    'drought_stress_kbdi_proxy_3mo', 'drought_stress_kbdi_proxy_6mo',
    'ndvi_cumdeficit_3mo', 'ndvi_cumdeficit_6mo', 'ndvi_cumdeficit_12mo',
    'ndvi_anom_delta', 'ndvi_drought_interact',
]
if new_feats_list:
    x_flat = np.concatenate([x_flat, np.concatenate(new_feats_list, axis=1)], axis=1)
    feature_names.extend(new_names_list)
    print(f'Added {len(new_names_list)} deep-lag/fallback features: {new_names_list}')
    new_feats_list, new_names_list = [], []

for _rc in _ROLLING_NPZ_COLS:
    _rc_l1 = f'{_rc}_lag1'
    if _rc in feature_names and _rc_l1 not in feature_names:
        _rc_2d = x_flat[:, feature_names.index(_rc)].reshape(N_MONTHS, N_PIXELS)
        _rc_l1_2d = np.zeros_like(_rc_2d)
        _rc_l1_2d[1:] = _rc_2d[:-1]
        _rc_l1_2d[0]  = _rc_2d[0]
        new_feats_list.append(_rc_l1_2d.reshape(-1, 1).astype(np.float32))
        new_names_list.append(_rc_l1)

if new_feats_list:
    x_flat = np.concatenate([x_flat, np.concatenate(new_feats_list, axis=1)], axis=1)
    feature_names.extend(new_names_list)
    print(f'Added {len(new_names_list)} lag-1 antecedent rolling features: {new_names_list}')

print(f'Total features : {len(feature_names)}  |  x_flat shape: {x_flat.shape}')

Keep only active pixels and drop near-constant features.


In [ ]:
VARIANCE_THRESHOLD  = 1e-10
BA_ACTIVE_THRESHOLD = 1e-8

ba_col_idx    = feature_names.index('burned_area_pred')
y_ba_flat     = x_flat[:, ba_col_idx].astype(np.float32)
y_flux_flat   = y_flat.astype(np.float32)

ba_all            = np.nan_to_num(y_ba_flat, nan=0.0).reshape(n_times, n_pixels)
active_pixel_2d   = np.nanmax(ba_all, axis=0) > BA_ACTIVE_THRESHOLD
active_pixel_flat = np.tile(active_pixel_2d, n_times)
valid_mask        = (~np.isnan(y_ba_flat)) & active_pixel_flat

n_raw = int(valid_mask.sum())
print(f'Step 1 — activity mask : {n_raw:,} rows  ({active_pixel_2d.sum():,} active pixels)')

# Variance filter on BA (the actual model target)
y_post_ba    = y_ba_flat[valid_mask]
pix_idx_post = (np.where(valid_mask)[0]) % n_pixels
unique_pix, inv = np.unique(pix_idx_post, return_inverse=True)
pix_var      = np.array([np.nanvar(y_post_ba[inv==k]) for k in range(len(unique_pix))])
row_var      = pix_var[inv]
final_mask   = np.zeros(len(y_ba_flat), dtype=bool)
ai           = np.where(valid_mask)[0]
final_mask[ai[row_var > VARIANCE_THRESHOLD]] = True

n_final = int(final_mask.sum())
print(f'Step 2 — variance filter: {n_final:,} rows kept, {n_raw-n_final:,} dropped')

x_valid      = x_flat[final_mask]
y_flux_valid = y_flux_flat[final_mask]
time_valid   = time_idx[final_mask]
lat_valid    = lat_flat[final_mask]
lon_valid    = lon_flat[final_mask]


Crop to the study region and define the regional masks.


In [ ]:
peat_idx      = feature_names.index('peat_mask')
is_peat_valid = x_valid[:, peat_idx] > 0.5

_pm_vals = x_valid[is_peat_valid, peat_idx]
print(f"[peat_mask diagnostic] within-Peat-region values: "
      f"min={_pm_vals.min():.3f} max={_pm_vals.max():.3f} "
      f"unique={len(np.unique(_pm_vals))} std={_pm_vals.std():.4f}")

ACTIVE_REGION = 'Maritime_West'
print(f'Modeling Active Region: {ACTIVE_REGION}')

idn_box      = (lat_valid >= -6.5) & (lat_valid <= 7.0) & (lon_valid >= 94.5) & (lon_valid <= 119.6)
mainland_box = (lat_valid > 7.0)   & (lon_valid <= 110.0)

# Island coordinate bounding boxes from fig_C_14_subregions (map_v32)
_is_java_valid          = (lat_valid >= -9.0)  & (lat_valid <= -5.7) & (lon_valid >= 105.0) & (lon_valid <= 114.7)
_is_bali_valid          = (lat_valid >= -9.0)  & (lat_valid <= -7.9) & (lon_valid >= 114.3) & (lon_valid <= 115.9)
_is_java_bali_valid     = _is_java_valid | _is_bali_valid
_is_sulawesi_valid      = (lat_valid >= -6.5)  & (lat_valid <= 1.9)  & (lon_valid >= 118.5) & (lon_valid <= 125.5)
_is_nusa_tenggara_valid = (lat_valid >= -11.2) & (lat_valid <= -6.8) & (lon_valid >= 114.8) & (lon_valid <= 127.5) & (~_is_java_bali_valid)
_is_maluku_valid        = (lat_valid >= -8.8)  & (lat_valid <= 3.2)  & (lon_valid >= 123.8) & (lon_valid <= 135.2) & (~_is_sulawesi_valid) & (~_is_nusa_tenggara_valid)
_is_peninsular_malaysia_valid = (lat_valid >= 1.0) & (lat_valid <= 7.0) & (lon_valid >= 100.0) & (lon_valid <= 104.5)
_phl_box_valid          = (lat_valid >= 5.0) & (lat_valid <= 21.5) & (lon_valid >= 116.5) & (lon_valid <= 127.5) & ~((lat_valid <= 7.5) & (lon_valid >= 116.0) & (lon_valid <= 119.6))
_core_minus_java_valid  = idn_box & (~_is_java_bali_valid) & (~_is_sulawesi_valid) & (~_is_peninsular_malaysia_valid) & (~_phl_box_valid)
is_kalimantan_peat_valid            = _core_minus_java_valid & is_peat_valid
is_sumatra_kalimantan_nonpeat_valid = _core_minus_java_valid & (~is_peat_valid)

# M3 fix (Aug 2026): Java_Bali, Sulawesi, and Nusa_Tenggara_Maluku are all
# NON-PEAT regions -- same as Sumatra_Kalimantan_NonPeat -- so they're moved
# out of the East federated loop and combined with Sumatra_Kalimantan_NonPeat
# into a single 'NonPeat_Combined' region trained in the West notebook.
# Philippines, PNG, and Peninsular Malaysia stay in East (topography/TC/
# human-access-driven fire regimes, mechanistically distinct from the
# equatorial ENSO/IOD moisture-deficit mechanism shared by the 4 non-peat
# regions above).

is_peat_nonpeat_valid = is_kalimantan_peat_valid | is_sumatra_kalimantan_nonpeat_valid
is_west_extra_nonpeat_valid = _is_java_bali_valid | _is_sulawesi_valid | (_is_nusa_tenggara_valid | _is_maluku_valid)
# The West domain: Peat + all 4 combined non-peat regions (Sumatra/Kalimantan
# non-peat, Java_Bali, Sulawesi, Nusa_Tenggara_Maluku)
is_west_domain_valid = is_peat_nonpeat_valid | is_west_extra_nonpeat_valid

masks = {
    'Indo_Peat':      is_peat_valid & idn_box,
    'Indo_NonPeat':   (~is_peat_valid) & idn_box,
    'Mainland':       mainland_box,
    'Rest_Of_SEAsia': ~(is_peat_valid & idn_box) & ~((~is_peat_valid) & idn_box) & ~(mainland_box),
    'Maritime_Continent': (~mainland_box) & (lat_valid <= 19.0),
    # West = Peat + the 4 combined non-peat regions (Sumatra/Kalimantan
    # non-peat, Java_Bali, Sulawesi, Nusa_Tenggara_Maluku) -- trained below as
    # 'Peat' and 'NonPeat_Combined'.
    'Maritime_West': (~mainland_box) & (lat_valid <= 19.0) & is_west_domain_valid,

    # East = the remaining Maritime-Continent island groupings (PNG,
    # Philippines, Peninsular Malaysia) -- fire regimes driven by topography/
    # TC/human-access/agricultural calendar rather than peat/ENSO-IOD drought.
    'Maritime_East': (~mainland_box) & (lat_valid <= 19.0) & (~is_west_domain_valid),
}

idn_mask = masks[ACTIVE_REGION]

x_idn_base  = x_valid[idn_mask].copy()
y_flux_idn  = y_flux_valid[idn_mask]
time_idn    = time_valid[idn_mask]
lat_idn     = lat_valid[idn_mask]
lon_idn     = lon_valid[idn_mask]
is_peat_idn = is_peat_valid[idn_mask]

pixel_coords             = np.stack([lat_idn, lon_idn], axis=1)
unique_pixels, pixel_inv = np.unique(pixel_coords, axis=0, return_inverse=True)
n_idn_pixels             = len(unique_pixels)
pix_lats                 = unique_pixels[:, 0]
pix_lons                 = unique_pixels[:, 1]

idn_mask_in_flat = np.zeros(len(x_flat), dtype=bool)
idn_mask_in_flat[np.where(final_mask)[0][idn_mask]] = True

YEAR_START  = 2003
YEAR_END    = 2020
TEST_OFFSET = 9
TRAIN_END_T = TEST_OFFSET * 12

print(f'Region : {ACTIVE_REGION}')
print(f'Rows   : {len(x_idn_base):,}  |  Pixels : {n_idn_pixels}')
print(f'Lat    : {lat_idn.min():.2f}-{lat_idn.max():.2f}  Lon : {lon_idn.min():.2f}-{lon_idn.max():.2f}')

DMI_BY_YEAR = {
    2003: 0.0,  2004: 0.2,  2005: 0.1,  2006: 0.9,  2007: 0.9,
    2008: 0.0,  2009: 0.1,  2010:-0.3,  2011: 0.1,  2012: 0.1,
    2013: 0.2,  2014: 0.4,  2015: 0.2,  2016: 0.0,  2017: 0.0,
    2018: 0.0,  2019: 1.2,  2020: 0.2,
}
_iod_yr_arr = YEAR_START + (time_idn // 12)
iod_idn = np.array([DMI_BY_YEAR.get(int(y), 0.0) for y in _iod_yr_arr], dtype=np.float32)
print(f'[F3] IOD index built: min={iod_idn.min():.2f} max={iod_idn.max():.2f}')
print(f'[M1] Per-region sub-domain flags (sub-region zones) built in Cell 5 below')

Extract the burned-area target (with unit correction).


In [ ]:
from scipy.stats import spearmanr

# ── P1-FIX: forward-fill NaN per pixel, chronologically ──────────────────────
ba_col_in_idn = feature_names.index('burned_area_pred')   # index in ORIGINAL feature_names

# Reshape BA to (n_times, n_pixels) on the full grid, fill NaN forward
ba_all_full = x_flat[:, ba_col_in_idn].reshape(n_times, n_pixels).copy()
for t in range(1, n_times):
    nan_mask = np.isnan(ba_all_full[t])
    ba_all_full[t][nan_mask] = ba_all_full[t-1][nan_mask]
ba_all_full = np.nan_to_num(ba_all_full, nan=0.0)

# ba_nan_flag: 1 for rows that were originally NaN (model can downweight)
ba_nan_orig   = x_flat[:, ba_col_in_idn].reshape(n_times, n_pixels)
ba_was_nan    = np.isnan(ba_nan_orig).reshape(-1).astype(np.float32)
ba_was_nan_idn = ba_was_nan[idn_mask_in_flat]

# Extract filled BA for IDN subset
ba_raw_idn = ba_all_full.reshape(-1)[idn_mask_in_flat].astype(np.float32)

# ── UNIT CORRECTION ────────────────────────────────────────────────────────────
# burned_area_pred / GFED4s y_total_flat is in km² per grid cell.
# Divide by pixel area to obtain a dimensionless burned fraction [0, 1].
#   Pixel area at 0.5° resolution (km²) = cos(lat) × (0.5 × 111.320)²
EARTH_DEG_KM   = 111.320
pixel_area_km2 = (np.cos(np.deg2rad(lat_idn)) *
                  ((round(float(np.median(np.diff(np.sort(np.unique(lat))))), 4) if len(np.unique(lat)) > 1 else 0.25) * EARTH_DEG_KM) ** 2).astype(np.float32)

y_ba_idn = np.maximum(0.0, ba_raw_idn / pixel_area_km2)

# ── Sanity checks ─────────────────────────────────────────────────────────────
print('── BA target extraction + unit correction ──────────────────────────────')
print(f'  Raw BA max         : {ba_raw_idn.max():.4f}  km² (GFED4s)')
print(f'  BA max (corrected) : {y_ba_idn.max():.6f}   ← must be ≤ 1.0')
print(f'  BA p99 fire months : {np.percentile(y_ba_idn[y_ba_idn>0], 99):.6f}')
print(f'  BA > 0             : {(y_ba_idn>0).mean()*100:.1f}%')
print(f'  NaN rows filled    : {int(ba_was_nan_idn.sum())}  (2020 partial year)')
print(f'  Pixel area (km²)   : {pixel_area_km2.min():.1f} – {pixel_area_km2.max():.1f}')

valid_both = (y_ba_idn > 0) & (y_flux_idn > 0)
if valid_both.sum() > 10:
    r, p = spearmanr(y_ba_idn[valid_both], y_flux_idn[valid_both])
    print(f'  Spearman r(BA,flux): {r:.3f}  (p={p:.2e})')
    if r < 0.15:
        raise RuntimeError(
            f'BA-flux Spearman r={r:.3f} is too low.\n'
            f'Check that GFED4s targets were loaded correctly in Cell 1.'
        )
    else:
        print('  BA-flux correlation OK ✓')

assert y_ba_idn.max() <= 1.0, (
    f'BA max={y_ba_idn.max():.4f} > 1 — unit correction failed.\n'
    f'Check lat_idn units: got {lat_idn[:3]}  (expected decimal degrees)')

print()
print('y_ba_idn validated ✓')

Engineer the regional predictors.


In [ ]:
import numpy as np

x_idn = x_idn_base.copy()
feature_names_ext = list(feature_names)

# Shared base predictors from .npz + GRACE + deep lags (Sec 2.2: exclude target, spatial coords, CAMS CH4)
BASE_EXCLUDE = [
    'burned_area_pred', 'peat_mask', 'ba_nan_flag', 'lat_flat', 'lon_flat',
    'cams_tc_ch4', 'gfed_defo_lag0', 'gfed_defo_cumsum_3mo_lag3', 'defo_ba_cumulative_9mo', 'crop_ba_cumulative_9mo', 'defo_ba_cumulative_12mo', 'crop_ba_cumulative_12mo',
]
PEAT_FEATURES_BA = [f for f in feature_names if f not in BASE_EXCLUDE]
print(f'Initial shared base features: {len(PEAT_FEATURES_BA)}')

# B1: Cumulative BA 13-24 month lag (Sec 2.2(iv))
ba_3d = ba_all_full.reshape(n_times, n_pixels)
cum_ba = np.zeros_like(ba_3d)
for t in range(n_times):
    t0 = max(0, t - 24); t1 = max(0, t - 12)
    if t1 > t0:
        cum_ba[t] = ba_3d[t0:t1].sum(axis=0)
x_idn = np.concatenate([x_idn, cum_ba.reshape(-1)[idn_mask_in_flat][:, None].astype(np.float32)], axis=1)
feature_names_ext.append('cumulative_ba_13_24mo'); PEAT_FEATURES_BA.append('cumulative_ba_13_24mo')

# B2: Previous calendar year total burned area (ba_prev_year, Sec 2.2(iv))
cal_years = YEAR_START + (time_idn // 12)
ba_prev_year = np.zeros(len(y_ba_idn), dtype=np.float32)
for pid in range(n_idn_pixels):
    pm = pixel_inv == pid
    yp_arr = y_ba_idn[pm]; yr_arr = cal_years[pm]
    ytot = {yr: float(np.sum(yp_arr[yr_arr == yr])) for yr in np.unique(yr_arr)}
    for idx in np.where(pm)[0]:
        ba_prev_year[idx] = ytot.get(cal_years[idx] - 1, 0.0)
x_idn = np.concatenate([x_idn, ba_prev_year[:, None].astype(np.float32)], axis=1)
feature_names_ext.append('ba_prev_year'); PEAT_FEATURES_BA.append('ba_prev_year')

# B3: Pixel Fire History strictly on pre-test training period (time_idn < TRAIN_END_T, Sec 2.2(iv))
fire_freq = np.zeros(n_idn_pixels, dtype=np.float32)
yrs_since = np.zeros(n_idn_pixels, dtype=np.float32)
for pid in range(n_idn_pixels):
    pm = pixel_inv == pid
    tr = pm & (time_idn < TRAIN_END_T)
    yb = y_ba_idn[tr]; tb = time_idn[tr]
    fm = tb[yb > 1e-6]
    fire_freq[pid] = len(fm) / max(tr.sum(), 1)
    yrs_since[pid] = (TRAIN_END_T - fm.max()) / 12.0 if len(fm) > 0 else 10.0
x_idn = np.concatenate([x_idn, fire_freq[pixel_inv][:, None], yrs_since[pixel_inv][:, None]], axis=1)
feature_names_ext += ['fire_freq_train', 'years_since_last_fire']
PEAT_FEATURES_BA  += ['fire_freq_train', 'years_since_last_fire']

# C: Deep soil-moisture deficit x positive ONI interaction (swvl3_deficit_x_oni, Sec 2.2)
if 'era5_swvl3_deficit_6mo' in feature_names_ext and 'oni_lag0' in feature_names_ext:
    interact = x_idn[:, feature_names_ext.index('era5_swvl3_deficit_6mo')] * np.maximum(0.0, x_idn[:, feature_names_ext.index('oni_lag0')])
    x_idn = np.concatenate([x_idn, interact[:, None].astype(np.float32)], axis=1)
    feature_names_ext.append('swvl3_deficit_x_oni'); PEAT_FEATURES_BA.append('swvl3_deficit_x_oni')
if 'era5_swvl3_deficit_6mo_lag1' in feature_names_ext and 'oni_lag1' in feature_names_ext:
    interact_l1 = x_idn[:, feature_names_ext.index('era5_swvl3_deficit_6mo_lag1')] * np.maximum(0.0, x_idn[:, feature_names_ext.index('oni_lag1')])
    x_idn = np.concatenate([x_idn, interact_l1[:, None].astype(np.float32)], axis=1)
    feature_names_ext.append('swvl3_deficit_x_oni_lag1'); PEAT_FEATURES_BA.append('swvl3_deficit_x_oni_lag1')

# E1: VPD cumulative deficit since January dry-season onset (+ lag1 antecedent version for forecaster, Sec 2.2(iii) & 2.3.1)
if 'era5_vpd' in feature_names:
    vpd_3d2 = x_flat[:, feature_names.index('era5_vpd')].reshape(n_times, len(lat), len(lon))
    cum_vpd = np.zeros_like(vpd_3d2)
    for yr in range(18):
        for m in range(12):
            t = yr * 12 + m
            if t >= n_times: break
            cum_vpd[t] = vpd_3d2[t] if m == 0 else cum_vpd[t - 1] + vpd_3d2[t]
    x_idn = np.concatenate([x_idn, cum_vpd.reshape(-1)[idn_mask_in_flat][:, None].astype(np.float32)], axis=1)
    feature_names_ext.append('vpd_cumdeficit_since_jan'); PEAT_FEATURES_BA.append('vpd_cumdeficit_since_jan')

    cum_vpd_l1 = np.zeros_like(cum_vpd)
    for yr in range(18):
        for m in range(12):
            t = yr * 12 + m
            if t >= n_times: break
            cum_vpd_l1[t] = 0.0 if (m == 0 or t == 0) else cum_vpd[t - 1]
    x_idn = np.concatenate([x_idn, cum_vpd_l1.reshape(-1)[idn_mask_in_flat][:, None].astype(np.float32)], axis=1)
    feature_names_ext.append('vpd_cumdeficit_since_jan_lag1'); PEAT_FEATURES_BA.append('vpd_cumdeficit_since_jan_lag1')

# E2: Consecutive months since last significant rainfall (<10th pre-test percentile, Sec 2.2(iii))
if 'era5_tp_lag0' in feature_names:
    tp_3d  = x_flat[:, feature_names.index('era5_tp_lag0')].reshape(n_times, len(lat), len(lon))
    tp_p10 = np.nanpercentile(tp_3d[:TRAIN_END_T], 10, axis=0)
    dsrain = np.zeros_like(tp_3d)
    for t in range(1, n_times):
        dsrain[t] = np.where(tp_3d[t] > tp_p10, 0, dsrain[t - 1] + 1)
    x_idn = np.concatenate([x_idn, dsrain.reshape(-1)[idn_mask_in_flat][:, None].astype(np.float32)], axis=1)
    feature_names_ext.append('days_since_last_rain'); PEAT_FEATURES_BA.append('days_since_last_rain')
    dsrain_l1 = np.zeros_like(dsrain); dsrain_l1[1:] = dsrain[:-1]
    x_idn = np.concatenate([x_idn, dsrain_l1.reshape(-1)[idn_mask_in_flat][:, None].astype(np.float32)], axis=1)
    feature_names_ext.append('days_since_last_rain_lag1'); PEAT_FEATURES_BA.append('days_since_last_rain_lag1')

# E3: Drought-breaker flag (two-month rainfall exceeding pre-test 90th percentile, Sec 2.2(iii))
if 'era5_tp_lag0' in feature_names_ext and 'era5_tp_lag1' in feature_names_ext:
    tp0 = x_idn[:, feature_names_ext.index('era5_tp_lag0')]
    tp1 = x_idn[:, feature_names_ext.index('era5_tp_lag1')]
    rr  = tp0 + tp1
    thresh = np.array([
        np.nanpercentile(rr[(pixel_inv == p) & (time_idn < TRAIN_END_T)], 90)
        if np.any((pixel_inv == p) & (time_idn < TRAIN_END_T)) else np.nanpercentile(rr[pixel_inv == p], 90)
        for p in range(n_idn_pixels)
    ], dtype=np.float32)
    db = (rr > thresh[pixel_inv]).astype(np.float32)
    x_idn = np.concatenate([x_idn, db[:, None]], axis=1)
    feature_names_ext.append('drought_breaker_flag'); PEAT_FEATURES_BA.append('drought_breaker_flag')

if 'era5_tp_lag1' in feature_names_ext and 'era5_tp_lag2' in feature_names_ext:
    tp1_ = x_idn[:, feature_names_ext.index('era5_tp_lag1')]
    tp2_ = x_idn[:, feature_names_ext.index('era5_tp_lag2')]
    rr_l1 = tp1_ + tp2_
    thresh_l1 = np.array([
        np.nanpercentile(rr_l1[(pixel_inv == p) & (time_idn < TRAIN_END_T)], 90)
        if np.any((pixel_inv == p) & (time_idn < TRAIN_END_T)) else np.nanpercentile(rr_l1[pixel_inv == p], 90)
        for p in range(n_idn_pixels)
    ], dtype=np.float32)
    db_l1 = (rr_l1 > thresh_l1[pixel_inv]).astype(np.float32)
    x_idn = np.concatenate([x_idn, db_l1[:, None]], axis=1)
    feature_names_ext.append('drought_breaker_flag_lag1'); PEAT_FEATURES_BA.append('drought_breaker_flag_lag1')

# E4: Monsoon-onset July-August precipitation timing (Sec 2.2(iii))
if 'era5_tp_lag1' in feature_names_ext:
    tp1_idx = feature_names_ext.index('era5_tp_lag1')
    _monsoon_mask = np.isin(time_idn % 12, [6, 7])
    monsoon_precip = x_idn[:, tp1_idx].copy()
    monsoon_precip[~_monsoon_mask] = 0.0
    x_idn = np.concatenate([x_idn, monsoon_precip[:, None].astype(np.float32)], axis=1)
    feature_names_ext.append('monsoon_onset_tp_sepoct'); PEAT_FEATURES_BA.append('monsoon_onset_tp_sepoct')

# E5: SOC domain z-score (pre-test stats) & soc_x_vpd_cumdeficit compound interaction (Sec 2.2)
if 'static_soc' in feature_names:
    sc = x_idn[:, feature_names.index('static_soc')].copy()
    _tr_rows = time_idn < TRAIN_END_T
    sc_z = ((sc - np.nanmean(sc[_tr_rows])) / (np.nanstd(sc[_tr_rows]) + 1e-8)).astype(np.float32)
    x_idn = np.concatenate([x_idn, sc_z[:, None]], axis=1)
    feature_names_ext.append('soc_domain_zscore'); PEAT_FEATURES_BA.append('soc_domain_zscore')
    if 'vpd_cumdeficit_since_jan' in feature_names_ext:
        fp = sc_z * np.maximum(0.0, x_idn[:, feature_names_ext.index('vpd_cumdeficit_since_jan')])
        x_idn = np.concatenate([x_idn, fp[:, None].astype(np.float32)], axis=1)
        feature_names_ext.append('soc_x_vpd_cumdeficit'); PEAT_FEATURES_BA.append('soc_x_vpd_cumdeficit')
    if 'vpd_cumdeficit_since_jan_lag1' in feature_names_ext:
        fp_l1 = sc_z * np.maximum(0.0, x_idn[:, feature_names_ext.index('vpd_cumdeficit_since_jan_lag1')])
        x_idn = np.concatenate([x_idn, fp_l1[:, None].astype(np.float32)], axis=1)
        feature_names_ext.append('soc_x_vpd_cumdeficit_lag1'); PEAT_FEATURES_BA.append('soc_x_vpd_cumdeficit_lag1')

# E6: Annual July DMI (iod_dmi_annual) & positive IOD x positive ONI interaction (Sec 2.2(ii))
x_idn = np.concatenate([x_idn, iod_idn[:, None]], axis=1)
feature_names_ext.append('iod_dmi_annual'); PEAT_FEATURES_BA.append('iod_dmi_annual')

if 'oni_lag0' in feature_names_ext:
    oni_idx_ext = feature_names_ext.index('oni_lag0')
    iod_x_oni = (np.maximum(0.0, iod_idn) * np.maximum(0.0, x_idn[:, oni_idx_ext])).astype(np.float32)
    x_idn = np.concatenate([x_idn, iod_x_oni[:, None]], axis=1)
    feature_names_ext.append('iod_x_oni'); PEAT_FEATURES_BA.append('iod_x_oni')

if 'oni_lag1' in feature_names_ext:
    oni1_idx_ext = feature_names_ext.index('oni_lag1')
    iod_x_oni_l1 = (np.maximum(0.0, iod_idn) * np.maximum(0.0, x_idn[:, oni1_idx_ext])).astype(np.float32)
    x_idn = np.concatenate([x_idn, iod_x_oni_l1[:, None]], axis=1)
    feature_names_ext.append('iod_x_oni_lag1'); PEAT_FEATURES_BA.append('iod_x_oni_lag1')

SHARED_BASE_FEATURES = list(PEAT_FEATURES_BA)

# ==============================================================================
# SUB-REGION MASKS (from fig_C_14_subregions / map_v32 bounding boxes)
# ==============================================================================
_core_idn_box   = (lat_idn >= -6.5) & (lat_idn <= 7.0) & (lon_idn >= 94.5) & (lon_idn <= 119.6)
_borneo_coast   = (lat_idn <= 7.5) & (lon_idn >= 116.0) & (lon_idn <= 119.6)
is_philippines  = ((lat_idn >= 5.0) & (lat_idn <= 21.5) & (lon_idn >= 116.5) & (lon_idn <= 127.5) & (~_borneo_coast)).astype(np.float32)
_is_java        = (lat_idn >= -9.0) & (lat_idn <= -5.7) & (lon_idn >= 105.0) & (lon_idn <= 114.7)
_is_bali        = (lat_idn >= -9.0) & (lat_idn <= -7.9) & (lon_idn >= 114.3) & (lon_idn <= 115.9)
is_java_bali    = (_is_java | _is_bali).astype(np.float32)
is_sulawesi     = ((lat_idn >= -6.5) & (lat_idn <= 1.9) & (lon_idn >= 118.5) & (lon_idn <= 125.5)).astype(np.float32)
_is_nusa_tenggara = (lat_idn >= -11.2) & (lat_idn <= -6.8) & (lon_idn >= 114.8) & (lon_idn <= 127.5) & (is_java_bali < 0.5)
_is_maluku        = (lat_idn >= -8.8) & (lat_idn <= 3.2) & (lon_idn >= 123.8) & (lon_idn <= 135.2) & (is_sulawesi < 0.5) & (~_is_nusa_tenggara)
is_nusa_tenggara_maluku = (_is_nusa_tenggara | _is_maluku).astype(np.float32)
is_png          = ((lat_idn >= -12.0) & (lat_idn <= 0.0) & (lon_idn > 130.0) & (~_is_maluku)).astype(np.float32)
is_peninsular_malaysia = ((lat_idn >= 1.0) & (lat_idn <= 7.0) & (lon_idn >= 100.0) & (lon_idn <= 104.5)).astype(np.float32)
_core_minus_java = _core_idn_box.astype(bool) & (is_java_bali < 0.5) & (is_sulawesi < 0.5) & (is_peninsular_malaysia < 0.5) & (is_philippines < 0.5)
is_kalimantan_peat = (_core_minus_java & is_peat_idn).astype(np.float32)
is_sumatra_kalimantan_nonpeat = (_core_minus_java & ~is_peat_idn.astype(bool)).astype(np.float32)
is_nonpeat_combined = np.clip(
    is_sumatra_kalimantan_nonpeat + is_java_bali + is_sulawesi + is_nusa_tenggara_maluku, 0.0, 1.0
).astype(np.float32)

# ==============================================================================
# PEAT-ONLY REGION-SPECIFIC PREDICTORS (Sec 2.2 line 21)
# Restricted strictly to the 'Peat' sub-region in the training loop;
# NonPeat_Monsoonal and NonPeat_Savanna receive ONLY SHARED_BASE_FEATURES.
# ==============================================================================
PEAT_ONLY_FEATURES = []

# 1. 6-month rolling and 12-month exponentially weighted rainfall-deficit memory terms (pre-test climatology)
if 'era5_tp_lag1' in feature_names:
    ROLL_DEFICIT_MONTHS = 6
    tp_l1_3d = x_flat[:, feature_names.index('era5_tp_lag1')].reshape(n_times, len(lat), len(lon))
    tp_l1_clim = np.nanmean(tp_l1_3d[:TRAIN_END_T], axis=0, keepdims=True)
    tp_l1_deficit = np.maximum(0.0, tp_l1_clim - tp_l1_3d)
    _csum = np.cumsum(tp_l1_deficit, axis=0)
    roll_deficit = np.zeros_like(tp_l1_deficit)
    roll_deficit[:ROLL_DEFICIT_MONTHS] = _csum[:ROLL_DEFICIT_MONTHS]
    roll_deficit[ROLL_DEFICIT_MONTHS:] = _csum[ROLL_DEFICIT_MONTHS:] - _csum[:-ROLL_DEFICIT_MONTHS]
    x_idn = np.concatenate([x_idn, roll_deficit.reshape(-1)[idn_mask_in_flat][:, None].astype(np.float32)], axis=1)
    feature_names_ext.append('tp_rolling_deficit_6mo_lag1'); PEAT_FEATURES_BA.append('tp_rolling_deficit_6mo_lag1')
    PEAT_ONLY_FEATURES.append('tp_rolling_deficit_6mo_lag1')

    EWMA_HALFLIFE_MONTHS = 12.0
    _ewma_decay = 0.5 ** (1.0 / EWMA_HALFLIFE_MONTHS)
    ewma_deficit = np.zeros_like(tp_l1_deficit)
    for t in range(1, n_times):
        ewma_deficit[t] = _ewma_decay * ewma_deficit[t - 1] + (1.0 - _ewma_decay) * tp_l1_deficit[t]
    x_idn = np.concatenate([x_idn, ewma_deficit.reshape(-1)[idn_mask_in_flat][:, None].astype(np.float32)], axis=1)
    feature_names_ext.append('tp_deficit_ewma_hl12mo_lag1'); PEAT_FEATURES_BA.append('tp_deficit_ewma_hl12mo_lag1')
    PEAT_ONLY_FEATURES.append('tp_deficit_ewma_hl12mo_lag1')

# 2. GRACE TWS month-over-month delta and cumulative drawdown since May (+ lag1 antecedent version for forecaster)
if 'grace_tws_lag0' in feature_names:
    _tws_3d = x_flat[:, feature_names.index('grace_tws_lag0')].reshape(n_times, n_pixels)
    _tws_delta = np.zeros_like(_tws_3d)
    _tws_delta[1:] = _tws_3d[1:] - _tws_3d[:-1]
    _tws_since_may = np.zeros_like(_tws_3d)
    for _yr in range(18):
        _may_t = _yr * 12 + 4
        for _m in range(12):
            _t = _yr * 12 + _m
            if _t < n_times and _m >= 4:
                _tws_since_may[_t] = _tws_3d[_may_t] - _tws_3d[_t]
    x_idn = np.concatenate([
        x_idn,
        _tws_delta.reshape(-1)[idn_mask_in_flat][:, None].astype(np.float32),
        _tws_since_may.reshape(-1)[idn_mask_in_flat][:, None].astype(np.float32),
    ], axis=1)
    feature_names_ext += ['grace_tws_delta', 'grace_tws_drawdown_since_may']
    PEAT_FEATURES_BA  += ['grace_tws_delta', 'grace_tws_drawdown_since_may']
    PEAT_ONLY_FEATURES += ['grace_tws_delta', 'grace_tws_drawdown_since_may']

    # Antecedent (t-1) reconstruction for lead-1 forecaster
    _tws_3d_l1 = np.zeros_like(_tws_3d)
    _tws_3d_l1[1:] = _tws_3d[:-1]; _tws_3d_l1[0] = _tws_3d[0]
    _tws_delta_l1 = np.zeros_like(_tws_3d_l1)
    _tws_delta_l1[1:] = _tws_3d_l1[1:] - _tws_3d_l1[:-1]
    _tws_since_may_l1 = np.zeros_like(_tws_3d_l1)
    for _yr in range(18):
        _may_t = _yr * 12 + 4
        for _m in range(12):
            _t = _yr * 12 + _m
            if _t < n_times and _m >= 4:
                _tws_since_may_l1[_t] = _tws_3d_l1[_may_t] - _tws_3d_l1[_t]
    x_idn = np.concatenate([
        x_idn,
        _tws_delta_l1.reshape(-1)[idn_mask_in_flat][:, None].astype(np.float32),
        _tws_since_may_l1.reshape(-1)[idn_mask_in_flat][:, None].astype(np.float32),
    ], axis=1)
    feature_names_ext += ['grace_tws_delta_lag1', 'grace_tws_drawdown_since_may_lag1']
    PEAT_FEATURES_BA  += ['grace_tws_delta_lag1', 'grace_tws_drawdown_since_may_lag1']
    PEAT_ONLY_FEATURES += ['grace_tws_delta_lag1', 'grace_tws_drawdown_since_may_lag1']

# 3. VPD anomaly restricted to active fire-season window (August-November, pre-test stats)
_fire_season_mask_mw = np.isin(time_idn % 12, [7, 8, 9, 10]).astype(np.float32)
for _v_src, _v_out in [('era5_vpd', 'vpd_anom_fireseason_zscore'),
                       ('era5_vpd_lag1', 'vpd_anom_fireseason_zscore_lag1')]:
    if _v_src in feature_names_ext:
        _vp = x_idn[:, feature_names_ext.index(_v_src)].astype(np.float32)
        _va = np.zeros_like(_vp)
        for _pid in range(n_idn_pixels):
            _pm = pixel_inv == _pid
            _tr = _pm & (time_idn < TRAIN_END_T)
            _ref = _tr if _tr.sum() >= 12 else _pm
            _mu, _sd = float(np.nanmean(_vp[_ref])), float(np.nanstd(_vp[_ref]))
            if _sd > 1e-8:
                _va[_pm] = (_vp[_pm] - _mu) / _sd
        _vfs = (np.maximum(0.0, _va) * _fire_season_mask_mw).astype(np.float32)
        x_idn = np.concatenate([x_idn, _vfs[:, None]], axis=1)
        feature_names_ext.append(_v_out); PEAT_FEATURES_BA.append(_v_out)
        PEAT_ONLY_FEATURES.append(_v_out)

# 4. Compound drought (moderate-plus El Nino & positive IOD) and continuous soft interaction
COMPOUND_ONI_THR, COMPOUND_IOD_THR = 1.0, 0.5
if 'oni_lag0' in feature_names_ext:
    _oni0 = x_idn[:, feature_names_ext.index('oni_lag0')]
    is_compound_drought = ((_oni0 >= COMPOUND_ONI_THR) & (iod_idn >= COMPOUND_IOD_THR)).astype(np.float32)
    x_idn = np.concatenate([x_idn, is_compound_drought[:, None]], axis=1)
    feature_names_ext.append('is_compound_drought'); PEAT_FEATURES_BA.append('is_compound_drought')
    PEAT_ONLY_FEATURES.append('is_compound_drought')
    if 'era5_swvl1_lag3' in feature_names_ext:
        _sw1_l3 = x_idn[:, feature_names_ext.index('era5_swvl1_lag3')]
        drought_oni_soft = (np.maximum(0.0, -_sw1_l3) * np.maximum(0.0, _oni0)).astype(np.float32)
        x_idn = np.concatenate([x_idn, drought_oni_soft[:, None]], axis=1)
        feature_names_ext.append('drought_oni_soft_interact'); PEAT_FEATURES_BA.append('drought_oni_soft_interact')
        PEAT_ONLY_FEATURES.append('drought_oni_soft_interact')

if 'oni_lag1' in feature_names_ext:
    _oni1 = x_idn[:, feature_names_ext.index('oni_lag1')]
    is_compound_drought_l1 = ((_oni1 >= COMPOUND_ONI_THR) & (iod_idn >= COMPOUND_IOD_THR)).astype(np.float32)
    x_idn = np.concatenate([x_idn, is_compound_drought_l1[:, None]], axis=1)
    feature_names_ext.append('is_compound_drought_lag1'); PEAT_FEATURES_BA.append('is_compound_drought_lag1')
    PEAT_ONLY_FEATURES.append('is_compound_drought_lag1')
    if 'era5_swvl1_lag3' in feature_names_ext:
        _sw1_l3 = x_idn[:, feature_names_ext.index('era5_swvl1_lag3')]
        drought_oni_soft_l1 = (np.maximum(0.0, -_sw1_l3) * np.maximum(0.0, _oni1)).astype(np.float32)
        x_idn = np.concatenate([x_idn, drought_oni_soft_l1[:, None]], axis=1)
        feature_names_ext.append('drought_oni_soft_interact_lag1'); PEAT_FEATURES_BA.append('drought_oni_soft_interact_lag1')
        PEAT_ONLY_FEATURES.append('drought_oni_soft_interact_lag1')

print(f'Peat-only region-specific features built ({len(PEAT_ONLY_FEATURES)}): {PEAT_ONLY_FEATURES}')

# ==============================================================================
# STRICT LEAD-1 REMOVAL OF CONCURRENT (LAG-0) FEATURES (Sec 2.3.1)
# ==============================================================================
import re as _re_cc

_LAG_TOKEN_RE = _re_cc.compile(r'lag(\d+)')
_TIME_VARYING_PREFIXES = (
    'era5_', 'ndvi_', 'evi_', 'oni', 'grace_', 'cams_', 'vpd', 'tp_', 'swvl',
    'drought_stress',
)
# Past-only by construction (built from >=1-month lagged inputs) although name has no lag token:
PAST_ONLY_WHITELIST = {
    'monsoon_onset_tp_sepoct',
    'precip_cumdeficit_3mo', 'precip_cumdeficit_6mo', 'precip_dryspell_months',
    'evi_cumdeficit_3mo', 'evi_cumdeficit_6mo', 'era5_vpd_cum3mo',
}
CONCURRENT_DERIVED_NAMES = {
    'vpd_spatial_lag',
    'vpd_cumdeficit_since_jan', 'vpd_cumdeficit_since_nov',
    'soc_x_vpd_cumdeficit',
    'days_since_last_rain',
    'drought_breaker_flag',
    'swvl3_deficit_x_oni', 'pm_swvl3_deficit_x_oni', 'png_drought_x_oni',
    'pm_inland_x_swvl3deficit', 'defo_x_dryseason',
    'pm_coastal_x_vpd',
    'pm_kbdi_x_oni', 'pm_kbdi_drought', 'png_kbdi_drought',
    'iod_x_oni', 'oni_x_dmi',
    'grace_tws_trend_6mo',
    
    'grace_tws_delta_mom', 'grace_tws_cum_drawdown_since_dry_season',
    'grace_tws_drawdown_x_peat', 'grace_tws_delta', 'grace_tws_drawdown_since_may',
    'vpd_anom_fireseason_zscore',
    'is_compound_drought', 'drought_oni_soft_interact',
}

def is_concurrent_feature(feat):
    f = feat.lower()
    if f in PAST_ONLY_WHITELIST:
        return False
    if f in CONCURRENT_DERIVED_NAMES:
        return True
    lags = [int(k) for k in _LAG_TOKEN_RE.findall(f)]
    if any(k == 0 for k in lags):
        return True
    if f.startswith(_TIME_VARYING_PREFIXES) and not any(k >= 1 for k in lags):
        return True
    return False

is_forbidden = is_concurrent_feature

def drop_concurrent_features(names, where='', label='', verbose=True):
    tag = where or label
    kept    = [n for n in names if not is_concurrent_feature(n)]
    dropped = [n for n in names if is_concurrent_feature(n)]
    if verbose:
        print(f'[lead-1 concurrent filter]{(" [" + tag + "]") if tag else ""} dropped {len(dropped)} of {len(names)}: {dropped}')
    return kept

drop_concurrent = drop_concurrent_features

def assert_no_concurrent(names, where='', label=''):
    tag = where or label
    bad = [n for n in names if is_concurrent_feature(n)]
    assert not bad, f'LEAD-1 LEAK GUARD{(" (" + tag + ")") if tag else ""}: {len(bad)} concurrent feature(s) reached model: {bad}'

PEAT_FEATURES_BA = drop_concurrent_features(PEAT_FEATURES_BA, where='candidate pool')
SHARED_BASE_FEATURES = drop_concurrent_features(SHARED_BASE_FEATURES, where='shared base')
PEAT_ONLY_FEATURES = drop_concurrent_features(PEAT_ONLY_FEATURES, where='Peat-only')
assert_no_concurrent(PEAT_FEATURES_BA, where='candidate pool')

PEAT_FEATURES_BA += ['y_lag1', 'y_lag3']
peat_feat_idx = [feature_names_ext.index(f) for f in PEAT_FEATURES_BA if f not in ('y_lag1', 'y_lag3')]
print(f'Feature matrix x_idn={x_idn.shape}, total features={len(feature_names_ext)}')

### Per-pixel cube-root normalization & quintiles fitted strictly on pre-test training period (2003-2011)


In [ ]:
SCALE_BA = 1e3
y_ba_cbrt = np.cbrt(y_ba_idn * SCALE_BA)
pixel_stats_ba = {}
y_ba_norm = np.zeros_like(y_ba_cbrt)
for pid in range(n_idn_pixels):
    pm = pixel_inv == pid
    tr_p = pm & (time_idn < TRAIN_END_T)
    ref_p = tr_p if tr_p.sum() >= 12 else pm
    m = float(np.nanmean(y_ba_cbrt[ref_p]))
    s = float(np.nanstd(y_ba_cbrt[ref_p])) + 1e-8
    pixel_stats_ba[pid] = (m, s)
    y_ba_norm[pm] = (y_ba_cbrt[pm] - m) / s
print(f'y_ba_norm range (pre-test 2003-2011 stats): [{y_ba_norm.min():.2f}, {y_ba_norm.max():.2f}]')

MIN_TRAIN_MONTHS = 24
_pix_means = np.array([
    np.mean(y_ba_idn[(pixel_inv == p) & (time_idn < TRAIN_END_T)])
    if np.any((pixel_inv == p) & (time_idn < TRAIN_END_T)) else np.mean(y_ba_idn[pixel_inv == p])
    for p in range(n_idn_pixels)
], dtype=np.float32)
_ba_q20, _ba_q40, _ba_q60, _ba_q80 = [float(np.quantile(_pix_means, q)) for q in [.2, .4, .6, .8]]
print(f'Pre-test quintile thresholds: Q20={_ba_q20:.6f} Q40={_ba_q40:.6f} Q60={_ba_q60:.6f} Q80={_ba_q80:.6f}')

Residualise the target on its lags and seed the lag buffers.


In [ ]:
y_lag1_global = np.zeros(len(y_ba_idn), dtype=np.float32)
y_lag3_global = np.zeros(len(y_ba_idn), dtype=np.float32)
for pid in range(n_idn_pixels):
    pm = pixel_inv == pid
    yp = y_ba_idn[pm]; tp = time_idn[pm]
    so = np.argsort(tp); ys = yp[so]
    l1 = np.zeros_like(ys); l1[1:] = ys[:-1]
    l3 = np.zeros_like(ys); l3[3:] = ys[:-3]
    oi = np.where(pm)[0][so]
    y_lag1_global[oi] = l1; y_lag3_global[oi] = l3

TEST_YEAR_START  = YEAR_START + TEST_OFFSET
_TRAIN_END_GLOBAL = (TEST_YEAR_START - YEAR_START) * 12
lag1_ols = {}; lag3_ols = {}
y_lag1_resid = y_lag1_global.copy().astype(np.float64)
y_lag3_resid = y_lag3_global.copy().astype(np.float64)

if 'oni_lag0' in feature_names_ext:
    oni_gi  = feature_names_ext.index('oni_lag0')
    oni_all = x_idn[:, oni_gi].astype(np.float64)
    for pid in range(n_idn_pixels):
        pm   = pixel_inv == pid; tr = pm & (time_idn < _TRAIN_END_GLOBAL)
        if tr.sum() < 12: lag1_ols[pid] = lag3_ols[pid] = (0.0, 0.0); continue
        X    = np.stack([np.ones(tr.sum()), oni_all[tr]], axis=1)
        for lag_g, resid_g, coef_d in [(y_lag1_global, y_lag1_resid, lag1_ols), (y_lag3_global, y_lag3_resid, lag3_ols)]:
            try: c, *_ = np.linalg.lstsq(X, lag_g[tr].astype(np.float64), rcond=None)
            except: c = [0.0, 0.0]
            # FIX 9 (ported from emulator): Only residualise if ONI explains < 30% of
            # the lag variance. When ONI explains most of the inter-annual fire
            # variance (R2 > 0.30), the residual has almost no signal left --
            # residualising actively harms the model by stripping out the main
            # predictable component.
            _lag_pred = c[0] + c[1] * oni_all[tr]
            _lag_obs  = lag_g[tr].astype(np.float64)
            _ss_tot   = float(np.var(_lag_obs)) * len(_lag_obs)
            _ss_res   = float(np.sum((_lag_obs - _lag_pred) ** 2))
            _r2_oni   = max(0.0, 1.0 - _ss_res / (_ss_tot + 1e-12))
            if _r2_oni > 0.30:
                coef_d[pid] = (0.0, 0.0)
            else:
                coef_d[pid] = (float(c[0]), float(c[1]))
            oni_px = oni_all[pm]
            c_eff = coef_d[pid]
            resid_g[pm] = lag_g[pm].astype(np.float64) - (c_eff[0] + c_eff[1]*oni_px)
    y_lag1_resid = y_lag1_resid.astype(np.float32)
    y_lag3_resid = y_lag3_resid.astype(np.float32)
    print(f'Lag residualisation: y_lag1 std raw={y_lag1_global.std():.4e} resid={y_lag1_resid.std():.4e}')
else:
    for pid in range(n_idn_pixels): lag1_ols[pid] = lag3_ols[pid] = (0.0, 0.0)
print('Lag residualisation complete ✓')


### Pooled Regional Architecture — Maritime West (Lead-1 Forecaster)

Aligned with Manuscript Sections 2.1, 2.2, and 2.3:
- 9 annual walk-forward folds (2012-2020), initial training window 2003-2011 (108 months)
- Fire-season-only evaluation window per sub-region + non-fire-season false-positive rate
- Two-stage pre-test feature selection: (1) complete-linkage Spearman |r|>0.70 clustering
retaining highest walk-forward permutation importance, (2) drop non-positive (<=0) importance
- Stratified 15% quintile pixel sample Optuna tuning maximizing de-normalized validation R2
- Per-region quantile loss alpha in [0.55, 0.65] for CatBoost & XGBoost; MSE for ExtraTrees
- sqrt(BA) magnitude-aware sample weights; inverse-validation-RMSE ensemble weights
- Per-fold hurdle threshold selection (0.05-0.60, step 0.05, precision >= 0.30, max val R2)
- Walk-forward OOF isotonic calibration with linear tail extrapolation (min 200 samples)
- Leak-free Jackknife+ 80% conformal prediction intervals from training-period OOF residuals
- 4 benchmark baselines (climatology, persistence, previous-year, OLS combination)


In [ ]:
import gc, sys, os, subprocess, pickle, warnings, multiprocessing
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.path as mpltPath
from scipy.stats import spearmanr
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, f1_score
from sklearn.inspection import permutation_importance
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import Ridge
from catboost import CatBoostRegressor, CatBoostClassifier
import xgboost as xgb

_N_CPU = max(1, multiprocessing.cpu_count())
try:
    _xgb_major = int(xgb.__version__.split('.')[0])
except Exception:
    _xgb_major = 1
if 'GPU_AVAILABLE' not in dir():
    try:
        GPU_AVAILABLE = subprocess.run(['nvidia-smi'], capture_output=True, text=True).returncode == 0
    except Exception:
        GPU_AVAILABLE = False

_CB_TASK_KW = dict(task_type='GPU', devices='0') if GPU_AVAILABLE else dict(task_type='CPU', thread_count=_N_CPU)
_XGB_DEV_KW = (dict(tree_method='hist', device='cuda') if _xgb_major >= 2
               else dict(tree_method='hist', device='cuda', predictor='gpu_predictor')) if GPU_AVAILABLE else dict(tree_method='hist', n_jobs=_N_CPU)

YEAR_START          = 2003
TEST_YEAR_START     = 2012
YEAR_END            = 2020
test_years          = list(range(2012, 2021))
EVAL_YEAR_START     = test_years[0]
EVAL_YEAR_END       = test_years[-1]
BA_ACTIVE_THRESHOLD = 1e-8
MIN_CALIB_SAMPLES   = 200

# ==============================================================================
# ISOTONIC CALIBRATION WITH LINEAR TAIL EXTRAPOLATION (Sec 2.3.4(iii))
# ==============================================================================
def _fit_isotonic_tail(x_fit, y_fit, tail_pct=95.0):
    x_fit = np.asarray(x_fit, dtype=np.float64)
    y_fit = np.asarray(y_fit, dtype=np.float64)
    iso_  = IsotonicRegression(out_of_bounds='clip', y_min=0.0)
    iso_.fit(x_fit, y_fit)
    ceiling   = float(np.max(x_fit))
    ceiling_y = float(iso_.predict([ceiling])[0])
    avg_slope = ceiling_y / max(ceiling, 1e-9)
    cut       = np.percentile(x_fit, tail_pct)
    tail_mask = x_fit >= cut
    if tail_mask.sum() >= 5 and len(np.unique(x_fit[tail_mask])) > 1:
        slope = float(np.polyfit(x_fit[tail_mask], y_fit[tail_mask], 1)[0])
        slope = float(np.clip(slope, 0.0, 3.0 * max(avg_slope, 1e-6)))
    else:
        slope = 0.0

    def _predict(x_arr):
        x_arr = np.asarray(x_arr, dtype=np.float64)
        out   = iso_.predict(np.minimum(x_arr, ceiling)).astype(np.float64)
        over  = x_arr > ceiling
        if np.any(over):
            out[over] = ceiling_y + slope * (x_arr[over] - ceiling)
        return np.maximum(0.0, out).astype(np.float32)
    return _predict

# ==============================================================================
# MAGNITUDE-AWARE SAMPLE WEIGHTS: sqrt(BA) normalized by mean (Sec 2.3.4(ii))
# ==============================================================================
def _sqrt_ba_weights(y_arr):
    y_arr = np.asarray(y_arr, dtype=np.float64)
    w = np.sqrt(np.maximum(y_arr, 0.0))
    m = float(w.mean())
    if m > 1e-12:
        w = w / m
    else:
        w = np.ones_like(y_arr)
    return np.maximum(w, 1e-3).astype(np.float32)

# ==============================================================================
# INVERSE-VALIDATION-RMSE ENSEMBLE WEIGHTS (Sec 2.3.3)
# ==============================================================================
def _inv_rmse_weights(y_true_val, p_cb_val, p_xgb_val, p_et_val):
    rmses = []
    for p in (p_cb_val, p_xgb_val, p_et_val):
        r = float(np.sqrt(mean_squared_error(y_true_val, p)))
        rmses.append(max(r, 1e-8))
    inv = np.array([1.0 / r for r in rmses], dtype=np.float64)
    return (inv / inv.sum()).astype(np.float32)

feature_names_idn_full = list(feature_names_ext)
for _nm, _arr in [
    ('lat_sin', np.sin(np.deg2rad(lat_idn)).astype(np.float32)),
    ('lat_cos', np.cos(np.deg2rad(lat_idn)).astype(np.float32)),
    ('lon_sin', np.sin(np.deg2rad(lon_idn)).astype(np.float32)),
    ('lon_cos', np.cos(np.deg2rad(lon_idn)).astype(np.float32)),
]:
    if _nm not in feature_names_ext:
        x_idn = np.concatenate([x_idn, _arr[:, None]], axis=1)
        feature_names_ext.append(_nm)
        feature_names_idn_full.append(_nm)
    if _nm not in SHARED_BASE_FEATURES:
        SHARED_BASE_FEATURES.append(_nm)
    if _nm not in PEAT_FEATURES_BA:
        PEAT_FEATURES_BA.append(_nm)

is_nonpeat_monsoonal = np.clip(is_sumatra_kalimantan_nonpeat + is_java_bali, 0.0, 1.0).astype(np.float32)
is_nonpeat_savanna   = np.clip(is_sulawesi + is_nusa_tenggara_maluku, 0.0, 1.0).astype(np.float32)
region_defs = {
    'Peat':              is_kalimantan_peat.astype(bool),
    'NonPeat_Monsoonal': is_nonpeat_monsoonal.astype(bool),
    'NonPeat_Savanna':   is_nonpeat_savanna.astype(bool),
}
SUBREGION_FIRE_SEASON_MONTHS = {
    'Peat':              [7, 8, 9, 10],  # Aug-Nov (Sec 2.1)
    'NonPeat_Monsoonal': [7, 8, 9, 10],  # Aug-Nov (Sec 2.1)
    'NonPeat_Savanna':   [7, 8, 9, 10],  # Aug-Nov (Sec 2.1)
}
FIRE_SEASON_MONTHS = [7, 8, 9, 10]

# Strict region-specific predictor assignment per Sec 2.2 line 21:
# Peat-specific features ONLY in 'Peat'; NonPeat_Monsoonal and NonPeat_Savanna use ONLY SHARED_BASE_FEATURES.
_BASE_MW = [f for f in SHARED_BASE_FEATURES if f in feature_names_ext and f not in ('y_lag1', 'y_lag3')]
_PEAT_MW = [f for f in (_BASE_MW + [p for p in PEAT_ONLY_FEATURES if p not in _BASE_MW]) if f in feature_names_ext]
REGION_FEAT_MAP = {
    'Peat':              _PEAT_MW,
    'NonPeat_Monsoonal': list(_BASE_MW),
    'NonPeat_Savanna':   list(_BASE_MW),
}
_FEAT_IDX_BASE  = [feature_names_ext.index(f) for f in _BASE_MW]
_ALL_FEAT_NAMES = feature_names_ext

# Precompute 1-month and 12-month observed BA lags for baseline comparisons and fuel-depletion rules
if 'y_lag1_global' not in dir():
    y_lag1_global = np.zeros(len(y_ba_idn), dtype=np.float32)
    y_lag3_global = np.zeros(len(y_ba_idn), dtype=np.float32)
    for _pid in range(n_idn_pixels):
        _pm = pixel_inv == _pid
        _yp = y_ba_idn[_pm]; _tp = time_idn[_pm]
        _so = np.argsort(_tp); _ys = _yp[_so]
        _l1 = np.zeros_like(_ys); _l1[1:] = _ys[:-1]
        _l3 = np.zeros_like(_ys); _l3[3:] = _ys[:-3]
        _oi = np.where(_pm)[0][_so]
        y_lag1_global[_oi] = _l1; y_lag3_global[_oi] = _l3

y_lag12_global = np.zeros(len(y_ba_idn), dtype=np.float32)
for _pid in range(n_idn_pixels):
    _pm = pixel_inv == _pid
    _yp = y_ba_idn[_pm]; _tp = time_idn[_pm]
    _so = np.argsort(_tp); _ys = _yp[_so]
    _l12 = np.zeros_like(_ys); _l12[12:] = _ys[:-12]
    _oi = np.where(_pm)[0][_so]
    y_lag12_global[_oi] = _l12

n_rows = len(y_ba_idn)
y_pred_final_ba     = np.full(n_rows, np.nan, dtype=np.float32)
y_pred_ba_prehurdle = np.full(n_rows, np.nan, dtype=np.float32)
y_pred_raw_ens      = np.full(n_rows, np.nan, dtype=np.float32)
y_pred_ba_ridge     = np.full(n_rows, np.nan, dtype=np.float32)
y_pred_ba_cb        = np.full(n_rows, np.nan, dtype=np.float32)
y_pred_ba_xgb       = np.full(n_rows, np.nan, dtype=np.float32)
y_pred_ba_et        = np.full(n_rows, np.nan, dtype=np.float32)
fire_prob_all       = np.full(n_rows, np.nan, dtype=np.float32)
y_pred_hurdle_prob  = np.full(n_rows, np.nan, dtype=np.float32)
y_fire_prob         = np.full(n_rows, np.nan, dtype=np.float32)
y_bin_all           = (y_ba_idn > BA_ACTIVE_THRESHOLD).astype(np.float32)
y_true_fire_bin     = np.full(n_rows, np.nan, dtype=np.float32)
y_pred_fire_bin     = np.full(n_rows, np.nan, dtype=np.float32)
y_used_hurdle       = np.zeros(n_rows, dtype=bool)
y_hurdle_zeroed     = np.zeros(n_rows, dtype=bool)
y_train_max_ba      = np.full(n_rows, np.nan, dtype=np.float32)
y_raw_ens_max_ba    = np.full(n_rows, np.nan, dtype=np.float32)

region_hurdle_gate              = {}
region_hurdle_gates             = {}
region_hyperparams              = {}
TUNED_HYPERPARAMS               = {}
region_feature_importance       = {}
REGION_FEATURE_IMPORTANCE       = {}
all_region_feat_importance      = {}
conformal_margins               = {}
conformal_margin_by_fold_domain = {}
non_fire_season_fpr             = {}

def _json_safe(v):
    if isinstance(v, (str, int, float, bool)) or v is None:
        return v
    return str(v)

print(f"\n{'='*80}")
print(" POOLED REGIONAL WALK-FORWARD TRAINING — MARITIME WEST (LEAD-1 FORECASTER)")
print(f"{'='*80}\n")

for reg_name, _reg_def in region_defs.items():
    row_mask = _reg_def[0] if isinstance(_reg_def, tuple) else _reg_def
    print(f"\n{'='*60}")
    print(f" REGION: {reg_name}")
    print(f"{'='*60}")

    n_reg_rows = int(row_mask.sum())
    if n_reg_rows < 50:
        print(f"[{reg_name}] Insufficient rows ({n_reg_rows}) — skipping.")
        continue

    reg_fire_months = SUBREGION_FIRE_SEASON_MONTHS[reg_name]
    cand_feats = [f for f in REGION_FEAT_MAP[reg_name] if f in _ALL_FEAT_NAMES]
    cand_feats = drop_concurrent_features(cand_feats, where=f'{reg_name} candidate pool', verbose=False)
    assert_no_concurrent(cand_feats, where=f'{reg_name} candidate pool')
    cand_idx = [_ALL_FEAT_NAMES.index(f) for f in cand_feats]
    global_rows = np.where(row_mask)[0]
    X_cand      = x_idn[row_mask][:, cand_idx].astype(np.float32)
    y_reg       = y_ba_idn[row_mask].astype(np.float32)
    t_reg       = time_idn[row_mask]
    pidinv_reg  = pixel_inv[row_mask]
    cal_yr_reg  = YEAR_START + (t_reg // 12)
    y_bin_reg   = (y_reg > BA_ACTIVE_THRESHOLD).astype(np.float32)
    _pretest_m  = cal_yr_reg < TEST_YEAR_START

    # ------------------------------------------------------------------
    # TWO-STAGE FEATURE SELECTION ON PRE-TEST TRAINING ROWS (Sec 2.2 para 5)
    # ------------------------------------------------------------------
    # 0. Drop constant/zero-variance features on pre-test training rows
    _col_std    = np.nanstd(X_cand[_pretest_m], axis=0)
    _active_pos = np.where(_col_std >= 1e-10)[0]
    cand_feats  = [cand_feats[i] for i in _active_pos]
    cand_idx    = [cand_idx[i] for i in _active_pos]
    X_cand      = X_cand[:, _active_pos]

    # Walk-forward permutation importance across pre-test held-out temporal validation folds (2009, 2010, 2011)
    _val_yrs_pre = [2009, 2010, 2011]
    imp_accum_pre = np.zeros(len(cand_feats), dtype=np.float64)
    imp_folds_pre = 0
    for _vyr in _val_yrs_pre:
        _tr_p = cal_yr_reg < _vyr
        _va_p = (cal_yr_reg == _vyr) & np.isin(t_reg % 12, reg_fire_months)
        if _va_p.sum() < 15:
            _va_p = (cal_yr_reg == _vyr)
        if _tr_p.sum() < 100 or _va_p.sum() < 10:
            continue
        _proxy = xgb.XGBRegressor(
            n_estimators=100, learning_rate=0.08, max_depth=4,
            subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0, **_XGB_DEV_KW
        )
        _proxy.fit(X_cand[_tr_p], y_reg[_tr_p])
        _perm_r = permutation_importance(
            _proxy, X_cand[_va_p], y_reg[_va_p], n_repeats=3, random_state=42, n_jobs=_N_CPU
        )
        imp_accum_pre += _perm_r.importances_mean
        imp_folds_pre += 1
    imp_mean_pre = (imp_accum_pre / imp_folds_pre) if imp_folds_pre > 0 else np.ones(len(cand_feats))

    # Stage 1: Complete-linkage hierarchical clustering on 1 - |Spearman r| at threshold 0.30 (|r| > 0.70)
    # strictly on pre-test training rows; retain single feature with highest walk-forward permutation importance
    _X_pre = X_cand[_pretest_m]
    _rng   = np.random.default_rng(42)
    _samp  = _rng.choice(len(_X_pre), size=min(8000, len(_X_pre)), replace=False)
    if len(cand_feats) > 2:
        _rho, _ = spearmanr(_X_pre[_samp], axis=0)
        _rho = np.atleast_2d(np.nan_to_num(_rho, nan=0.0, posinf=1.0, neginf=-1.0))
        np.fill_diagonal(_rho, 1.0)
        _dist = np.clip(1.0 - np.abs(np.clip(_rho, -1.0, 1.0)), 0.0, 2.0)
        _dist = (_dist + _dist.T) / 2.0
        np.fill_diagonal(_dist, 0.0)
        _link = linkage(squareform(_dist, checks=False), method='complete')
        _clusters = fcluster(_link, 0.30, criterion='distance')
    else:
        _clusters = np.arange(len(cand_feats))

    _stage1_pos = []
    for _cid in np.unique(_clusters):
        _c_members = np.where(_clusters == _cid)[0]
        _best_m = _c_members[int(np.argmax(imp_mean_pre[_c_members]))]
        _stage1_pos.append(int(_best_m))
    _stage1_pos = sorted(_stage1_pos)

    # Stage 2: Remove surviving features with non-positive (<= 0) mean walk-forward permutation importance
    _stage2_pos = [i for i in _stage1_pos if imp_mean_pre[i] > 0.0]
    if len(_stage2_pos) < 5:
        _stage2_pos = sorted(_stage1_pos, key=lambda i: imp_mean_pre[i], reverse=True)[:max(5, len(_stage1_pos))]
        _stage2_pos = sorted(_stage2_pos)

    reg_feats    = [cand_feats[i] for i in _stage2_pos]
    reg_feat_idx = [cand_idx[i] for i in _stage2_pos]
    merged_feats = list(reg_feats)
    reg_feat_names = list(reg_feats)
    X_reg        = X_cand[:, _stage2_pos].astype(np.float32)
    print(f"[{reg_name}] Two-stage feature selection (pre-test 2003-2011): "
          f"{len(cand_feats)} candidates -> {len(_stage1_pos)} after Spearman |r|>0.70 -> "
          f"{len(reg_feats)} after >0 permutation importance.")

    # ------------------------------------------------------------------
    # PER-SUB-REGION ASYMMETRIC QUANTILE LOSS ALPHA IN [0.55, 0.65] (Sec 2.3.3 & 2.3.4(i))
    # ------------------------------------------------------------------
    _active_frac_reg      = float((y_reg[_pretest_m] > BA_ACTIVE_THRESHOLD).mean())
    REGION_QUANTILE_ALPHA = float(np.clip(0.65 - 0.5 * _active_frac_reg, 0.55, 0.65))
    print(f"[{reg_name}] Pre-test active-fire fraction={_active_frac_reg:.3f} -> Quantile alpha={REGION_QUANTILE_ALPHA:.2f}")

    # ------------------------------------------------------------------
    # OPTUNA HYPERPARAMETER TUNING ON 15% STRATIFIED QUINTILE PIXEL SAMPLE (Sec 2.3.3)
    # Maximizing validation R2 on de-normalized burned-area targets across pre-test folds
    # ------------------------------------------------------------------
    _unique_p = np.unique(pidinv_reg[_pretest_m])
    _p_means  = np.array([
        float(np.mean(y_reg[(pidinv_reg == p) & _pretest_m])) for p in _unique_p
    ], dtype=np.float32)
    _q_edges  = np.quantile(_p_means, [0.2, 0.4, 0.6, 0.8]) if len(_unique_p) >= 5 else np.array([0.0])
    _p_bins   = np.digitize(_p_means, _q_edges)
    n_samp    = max(15, int(np.ceil(len(_unique_p) * 0.15)))
    if len(_unique_p) > n_samp:
        _sampled_p = []
        for _b in range(5):
            _bp = _unique_p[_p_bins == _b]
            if len(_bp) == 0:
                continue
            _nb = max(1, int(np.round(n_samp * len(_bp) / len(_unique_p))))
            _sampled_p.append(_rng.choice(_bp, size=min(len(_bp), _nb), replace=False))
        _s_p = np.concatenate(_sampled_p) if _sampled_p else _unique_p
    else:
        _s_p = _unique_p

    _opt_rows = _pretest_m & np.isin(pidinv_reg, _s_p)
    X_opt     = X_reg[_opt_rows]
    y_opt     = y_reg[_opt_rows]
    yr_opt    = cal_yr_reg[_opt_rows]
    mo_opt    = (t_reg[_opt_rows] % 12)
    opt_w     = _sqrt_ba_weights(y_opt)

    _opt_splits = []
    for _vyr in [2009, 2010, 2011]:
        _tr_i = np.where(yr_opt < _vyr)[0]
        _va_i = np.where((yr_opt == _vyr) & np.isin(mo_opt, reg_fire_months))[0]
        if len(_va_i) < 10:
            _va_i = np.where(yr_opt == _vyr)[0]
        if len(_tr_i) >= 50 and len(_va_i) >= 5 and np.var(y_opt[_va_i]) > 1e-16:
            _opt_splits.append((_tr_i, _va_i))

    def cb_obj(trial):
        p = dict(
            iterations=200,
            depth=trial.suggest_int('depth', 4, 8),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 1.0, 30.0),
            loss_function=f'Quantile:alpha={REGION_QUANTILE_ALPHA}',
            random_seed=42, verbose=0, **_CB_TASK_KW,
        )
        if not _opt_splits:
            return 0.0
        scores = []
        for _tr_i, _va_i in _opt_splits:
            m = CatBoostRegressor(**p).fit(X_opt[_tr_i], y_opt[_tr_i], sample_weight=opt_w[_tr_i])
            pred = np.maximum(0.0, m.predict(X_opt[_va_i]).astype(np.float32))
            scores.append(float(r2_score(y_opt[_va_i], pred)))
        return float(np.mean(scores))

    def xgb_obj(trial):
        p = dict(
            n_estimators=150,
            max_depth=trial.suggest_int('max_depth', 4, 8),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            subsample=trial.suggest_float('subsample', 0.6, 1.0),
            colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
            objective='reg:quantileerror', quantile_alpha=REGION_QUANTILE_ALPHA,
            random_state=42, verbosity=0, **_XGB_DEV_KW,
        )
        if not _opt_splits:
            return 0.0
        scores = []
        for _tr_i, _va_i in _opt_splits:
            m = xgb.XGBRegressor(**p).fit(X_opt[_tr_i], y_opt[_tr_i], sample_weight=opt_w[_tr_i])
            pred = np.maximum(0.0, m.predict(X_opt[_va_i]).astype(np.float32))
            scores.append(float(r2_score(y_opt[_va_i], pred)))
        return float(np.mean(scores))

    nt = max(10, min(25, int(len(_unique_p) / 40) + 10))
    cb_study = optuna.create_study(direction='maximize')
    cb_study.optimize(cb_obj, n_trials=nt)
    CB_PARAMS = dict(
        cb_study.best_params, iterations=1000,
        loss_function=f'Quantile:alpha={REGION_QUANTILE_ALPHA}',
        random_seed=42, verbose=0, **_CB_TASK_KW,
    )

    xgb_study = optuna.create_study(direction='maximize')
    xgb_study.optimize(xgb_obj, n_trials=nt)
    XGB_PARAMS = dict(
        xgb_study.best_params, n_estimators=600,
        objective='reg:quantileerror', quantile_alpha=REGION_QUANTILE_ALPHA,
        random_state=42, verbosity=0, **_XGB_DEV_KW,
    )
    ET_PARAMS = dict(n_estimators=400, max_depth=None, min_samples_leaf=20, n_jobs=_N_CPU, random_state=42)

    # ------------------------------------------------------------------
    # HELPER: SELECT HURDLE THRESHOLD (0.05..0.60, step 0.05, precision >= 0.30, max val R2, Sec 2.3.2)
    # ------------------------------------------------------------------
    def _select_hurdle_threshold(probs_val, ens_val, y_val, yb_val):
        cand_grid = np.round(np.arange(0.05, 0.65, 0.05), 2)
        valid_cands = []
        best_f2_fallback, best_thr_fallback = -1.0, 0.30
        for thr in cand_grid:
            yh   = (probs_val >= thr).astype(int)
            tp   = int(np.sum((yh == 1) & (yb_val == 1)))
            fp   = int(np.sum((yh == 1) & (yb_val == 0)))
            fn   = int(np.sum((yh == 0) & (yb_val == 1)))
            prec = tp / max(tp + fp, 1)
            rec  = tp / max(tp + fn, 1)
            f2   = (5.0 * prec * rec) / max(4.0 * prec + rec, 1e-9)
            if f2 > best_f2_fallback:
                best_f2_fallback, best_thr_fallback = f2, float(thr)
            if prec >= 0.30:
                gated = np.where(probs_val >= thr, ens_val, 0.0)
                val_r2 = float(r2_score(y_val, gated)) if len(y_val) > 1 and np.var(y_val) > 1e-16 else -1e9
                valid_cands.append((val_r2, f2, float(thr)))
        if valid_cands:
            valid_cands.sort(key=lambda x: (x[0], x[1]), reverse=True)
            return valid_cands[0][2], valid_cands[0][1]
        return best_thr_fallback, max(0.0, best_f2_fallback)

    # ------------------------------------------------------------------
    # WALK-FORWARD POOLED TRAINING LOOP (2012-2020, 9 FOLDS)
    # ------------------------------------------------------------------
    y_pred_reg           = np.full(n_reg_rows, np.nan, dtype=np.float32)
    y_pred_prehurdle_reg = np.full(n_reg_rows, np.nan, dtype=np.float32)
    y_pred_raw_reg       = np.full(n_reg_rows, np.nan, dtype=np.float32)
    y_cb_reg             = np.full(n_reg_rows, np.nan, dtype=np.float32)
    y_xgb_reg            = np.full(n_reg_rows, np.nan, dtype=np.float32)
    y_et_reg             = np.full(n_reg_rows, np.nan, dtype=np.float32)
    fire_prob_reg        = np.full(n_reg_rows, np.nan, dtype=np.float32)
    pred_fire_bin_reg    = np.full(n_reg_rows, np.nan, dtype=np.float32)
    used_hurdle_reg      = np.zeros(n_reg_rows, dtype=bool)
    hurdle_zeroed_reg    = np.zeros(n_reg_rows, dtype=bool)
    train_max_reg        = np.full(n_reg_rows, np.nan, dtype=np.float32)
    raw_ens_max_reg      = np.full(n_reg_rows, np.nan, dtype=np.float32)

    conf_margins     = {}
    imp_accum_f      = np.zeros(len(reg_feats), dtype=np.float64)
    imp_weight       = 0
    _calib_pred_hist = []
    _calib_true_hist = []
    _oof_resid_hist  = []
    _fold_gates      = []
    _fold_f2s        = []
    _nonfire_fp_cnt  = 0
    _nonfire_tot_cnt = 0

    # Pre-seed out-of-fold calibration & conformal residual history on pre-test split (2003-2009 -> 2010)
    _seed_tr = cal_yr_reg < 2010
    _seed_va = (cal_yr_reg == 2010) & np.isin(t_reg % 12, reg_fire_months)
    if _seed_va.sum() < 15:
        _seed_va = (cal_yr_reg == 2010)
    if _seed_tr.sum() >= 100 and _seed_va.sum() >= 10:
        _sw_s  = _sqrt_ba_weights(y_reg[_seed_tr])
        _cb_s  = CatBoostRegressor(**CB_PARAMS).fit(X_reg[_seed_tr], y_reg[_seed_tr], sample_weight=_sw_s)
        _xgb_s = xgb.XGBRegressor(**XGB_PARAMS).fit(X_reg[_seed_tr], y_reg[_seed_tr], sample_weight=_sw_s)
        _et_s  = ExtraTreesRegressor(**ET_PARAMS).fit(X_reg[_seed_tr], y_reg[_seed_tr], sample_weight=_sw_s)
        _p_cb_s  = np.maximum(0.0, _cb_s.predict(X_reg[_seed_va]).astype(np.float32))
        _p_xgb_s = np.maximum(0.0, _xgb_s.predict(X_reg[_seed_va]).astype(np.float32))
        _p_et_s  = np.maximum(0.0, _et_s.predict(X_reg[_seed_va]).astype(np.float32))
        _w_s     = _inv_rmse_weights(y_reg[_seed_va], _p_cb_s, _p_xgb_s, _p_et_s)
        _ens_s   = _w_s[0] * _p_cb_s + _w_s[1] * _p_xgb_s + _w_s[2] * _p_et_s
        _calib_pred_hist.append(_ens_s.copy())
        _calib_true_hist.append(y_reg[_seed_va].copy())
        _oof_resid_hist.append(np.abs(y_reg[_seed_va] - _ens_s).astype(np.float32))

    for t_yr in test_years:
        tr_mask         = cal_yr_reg < t_yr
        te_mask         = (cal_yr_reg == t_yr) & np.isin(t_reg % 12, reg_fire_months)
        nonfire_te_mask = (cal_yr_reg == t_yr) & (~np.isin(t_reg % 12, reg_fire_months))
        if tr_mask.sum() < 100 or te_mask.sum() == 0:
            continue

        # Per-fold leak-free recomputation of fire_freq_train and years_since_last_fire using cal_yr_reg < t_yr (Sec 2.2)
        if 'fire_freq_train' in reg_feats or 'years_since_last_fire' in reg_feats:
            _t_cut_f = (t_yr - YEAR_START) * 12
            _ff_col = reg_feats.index('fire_freq_train') if 'fire_freq_train' in reg_feats else None
            _ys_col = reg_feats.index('years_since_last_fire') if 'years_since_last_fire' in reg_feats else None
            for _pid_f in np.unique(pidinv_reg):
                _pm_f = (pidinv_reg == _pid_f)
                _tr_f = _pm_f & tr_mask
                _fm_f = t_reg[_tr_f][y_reg[_tr_f] > BA_ACTIVE_THRESHOLD]
                if _ff_col is not None:
                    X_reg[_pm_f, _ff_col] = float(len(_fm_f)) / max(int(_tr_f.sum()), 1)
                if _ys_col is not None:
                    X_reg[_pm_f, _ys_col] = float(_t_cut_f - _fm_f.max()) / 12.0 if len(_fm_f) > 0 else 10.0

        # Inner temporal validation split (year t_yr - 1) for inverse-RMSE weights,
        # hurdle threshold selection (Sec 2.3.2 & 2.3.3), and OOF calibration/residual accumulation
        _val_yr  = t_yr - 1
        _in_tr   = cal_yr_reg < _val_yr
        _in_va   = (cal_yr_reg == _val_yr) & np.isin(t_reg % 12, reg_fire_months)
        if _in_va.sum() < 15:
            _in_va = (cal_yr_reg == _val_yr)
        if _in_tr.sum() >= 100 and _in_va.sum() >= 10:
            _sw_in  = _sqrt_ba_weights(y_reg[_in_tr])
            _cb_in  = CatBoostRegressor(**CB_PARAMS).fit(X_reg[_in_tr], y_reg[_in_tr], sample_weight=_sw_in)
            _xgb_in = xgb.XGBRegressor(**XGB_PARAMS).fit(X_reg[_in_tr], y_reg[_in_tr], sample_weight=_sw_in)
            _et_in  = ExtraTreesRegressor(**ET_PARAMS).fit(X_reg[_in_tr], y_reg[_in_tr], sample_weight=_sw_in)
            _p_cb_v  = np.maximum(0.0, _cb_in.predict(X_reg[_in_va]).astype(np.float32))
            _p_xgb_v = np.maximum(0.0, _xgb_in.predict(X_reg[_in_va]).astype(np.float32))
            _p_et_v  = np.maximum(0.0, _et_in.predict(X_reg[_in_va]).astype(np.float32))
            w_ens    = _inv_rmse_weights(y_reg[_in_va], _p_cb_v, _p_xgb_v, _p_et_v)
            _ens_v   = w_ens[0] * _p_cb_v + w_ens[1] * _p_xgb_v + w_ens[2] * _p_et_v

            _clf_in = CatBoostClassifier(
                iterations=400, depth=5, learning_rate=0.08,
                random_seed=42, verbose=0, class_weights=[1.0, 2.5], **_CB_TASK_KW
            )
            if y_bin_reg[_in_tr].sum() > 0 and y_bin_reg[_in_tr].sum() < _in_tr.sum():
                _clf_in.fit(X_reg[_in_tr], y_bin_reg[_in_tr])
                _prob_v = _clf_in.predict_proba(X_reg[_in_va])[:, 1].astype(np.float32)
            else:
                _prob_v = np.zeros(_in_va.sum(), dtype=np.float32)

            HURDLE_GATE_FOLD, fold_f2 = _select_hurdle_threshold(_prob_v, _ens_v, y_reg[_in_va], y_bin_reg[_in_va])
            _calib_pred_hist.append(_ens_v.copy())
            _calib_true_hist.append(y_reg[_in_va].copy())
            _gated_v = np.where(_prob_v >= HURDLE_GATE_FOLD, _ens_v, 0.0)
            _oof_resid_hist.append(np.abs(y_reg[_in_va] - _gated_v).astype(np.float32))
        else:
            w_ens = np.array([1.0 / 3.0, 1.0 / 3.0, 1.0 / 3.0], dtype=np.float32)
            HURDLE_GATE_FOLD, fold_f2 = 0.30, 0.0

        _fold_gates.append(HURDLE_GATE_FOLD)
        _fold_f2s.append(fold_f2)

        # Fit full fold models on all training data (cal_yr_reg < t_yr)
        X_tr  = X_reg[tr_mask];  y_tr  = y_reg[tr_mask]
        X_te  = X_reg[te_mask];  y_te  = y_reg[te_mask]
        yb_tr = y_bin_reg[tr_mask]
        sample_w = _sqrt_ba_weights(y_tr)

        cb   = CatBoostRegressor(**CB_PARAMS).fit(X_tr, y_tr, sample_weight=sample_w)
        xgbm = xgb.XGBRegressor(**XGB_PARAMS).fit(X_tr, y_tr, sample_weight=sample_w)
        if len(np.unique(y_tr)) > 2:
            et = ExtraTreesRegressor(**ET_PARAMS).fit(X_tr, y_tr, sample_weight=sample_w)
            p_et_te = np.maximum(0.0, et.predict(X_te).astype(np.float32))
        else:
            et = None
            p_et_te = np.zeros(len(X_te), dtype=np.float32)

        p_cb_te  = np.maximum(0.0, cb.predict(X_te).astype(np.float32))
        p_xgb_te = np.maximum(0.0, xgbm.predict(X_te).astype(np.float32))
        ens_te   = w_ens[0] * p_cb_te + w_ens[1] * p_xgb_te + w_ens[2] * p_et_te

        # Post-hoc isotonic calibration from accumulated prior walk-forward OOF folds (min 200 samples, Sec 2.3.4(iii))
        _calib_n = sum(len(a) for a in _calib_pred_hist)
        if _calib_n >= MIN_CALIB_SAMPLES:
            _cal_x = np.concatenate(_calib_pred_hist)
            _cal_y = np.concatenate(_calib_true_hist)
            iso_predict    = _fit_isotonic_tail(_cal_x, _cal_y)
            ens_calibrated = iso_predict(ens_te)
            calib_status   = f"on(n={_calib_n})"
        else:
            iso_predict    = None
            ens_calibrated = ens_te.copy()
            calib_status   = f"off(n={_calib_n}<{MIN_CALIB_SAMPLES})"

        # Hurdle classifier
        clf_fold = CatBoostClassifier(
            iterations=400, depth=5, learning_rate=0.08,
            random_seed=42, verbose=0, class_weights=[1.0, 2.5], **_CB_TASK_KW
        )
        if yb_tr.sum() > 0 and yb_tr.sum() < len(yb_tr):
            clf_fold.fit(X_tr, yb_tr)
            fire_prob = clf_fold.predict_proba(X_te)[:, 1].astype(np.float32)
        else:
            fire_prob = np.zeros(len(X_te), dtype=np.float32)

        ens_gated = ens_calibrated.copy()
        ens_gated[fire_prob < HURDLE_GATE_FOLD] = 0.0

        # Process-informed post-processing rules — Maritime West ONLY (Sec 2.3.3):
        # 1. La Nina suppression (ONI <= -0.5): multiply by 0.05 in Peat, set to 0.0 in non-peat
        _oni_col_name = 'oni_lag1' if 'oni_lag1' in _ALL_FEAT_NAMES else 'oni_lag1'
        oni_te = x_idn[row_mask][te_mask, _ALL_FEAT_NAMES.index(_oni_col_name)].astype(np.float32)
        lanina_te = (oni_te <= -0.5)
        lanina_scale = 0.05 if reg_name == 'Peat' else 0.0
        ens_gated[lanina_te] *= lanina_scale

        # 2. Fuel-depletion damping: multiply by 0.4 when previous month's burned area > 0.015 km2 (Peat) or 0.005 km2 (non-peat)
        prev_month_ba_km2_te = (y_lag1_global[row_mask][te_mask] * pixel_area_km2[row_mask][te_mask]).astype(np.float32)
        fuel_thr_km2 = 0.015 if reg_name == 'Peat' else 0.005
        ens_gated[prev_month_ba_km2_te > fuel_thr_km2] *= 0.4

        # Non-fire-season false-positive rate evaluation (Sec 2.1)
        if nonfire_te_mask.sum() > 0:
            X_nf = X_reg[nonfire_te_mask]
            y_nf = y_reg[nonfire_te_mask]
            if yb_tr.sum() > 0 and yb_tr.sum() < len(yb_tr):
                prob_nf = clf_fold.predict_proba(X_nf)[:, 1].astype(np.float32)
            else:
                prob_nf = np.zeros(len(X_nf), dtype=np.float32)
            p_cb_nf  = np.maximum(0.0, cb.predict(X_nf).astype(np.float32))
            p_xgb_nf = np.maximum(0.0, xgbm.predict(X_nf).astype(np.float32))
            p_et_nf  = np.maximum(0.0, et.predict(X_nf).astype(np.float32)) if et is not None else np.zeros(len(X_nf), dtype=np.float32)
            ens_nf   = w_ens[0] * p_cb_nf + w_ens[1] * p_xgb_nf + w_ens[2] * p_et_nf
            if iso_predict is not None:
                ens_nf = iso_predict(ens_nf)
            ens_nf[prob_nf < HURDLE_GATE_FOLD] = 0.0
            _nonfire_fp_cnt  += int(np.sum((ens_nf > BA_ACTIVE_THRESHOLD) & (y_nf <= BA_ACTIVE_THRESHOLD)))
            _nonfire_tot_cnt += int(np.sum(y_nf <= BA_ACTIVE_THRESHOLD))

        y_pred_reg[te_mask]           = ens_gated
        y_pred_prehurdle_reg[te_mask] = ens_calibrated
        y_pred_raw_reg[te_mask]       = ens_te
        y_cb_reg[te_mask]             = p_cb_te
        y_xgb_reg[te_mask]            = p_xgb_te
        y_et_reg[te_mask]             = p_et_te
        fire_prob_reg[te_mask]        = fire_prob
        pred_fire_bin_reg[te_mask]    = (fire_prob >= HURDLE_GATE_FOLD).astype(np.float32)
        used_hurdle_reg[te_mask]      = True
        hurdle_zeroed_reg[te_mask]    = (fire_prob < HURDLE_GATE_FOLD)
        train_max_reg[te_mask]        = float(np.nanmax(y_tr)) if len(y_tr) > 0 else 0.0
        raw_ens_max_reg[te_mask]      = float(np.nanmax(ens_te)) if len(ens_te) > 0 else 0.0

        imp_accum_f += np.array(cb.get_feature_importance()) * te_mask.sum()
        imp_weight  += te_mask.sum()

        # Leak-free Jackknife+ 80% conformal margin from training-period OOF residuals only (< t_yr, Sec 2.3.5)
        if _oof_resid_hist:
            abs_resid_tr = np.concatenate(_oof_resid_hist)
        else:
            p_tr_ens = w_ens[0] * np.maximum(0.0, cb.predict(X_tr).astype(np.float32)) + \
                       w_ens[1] * np.maximum(0.0, xgbm.predict(X_tr).astype(np.float32))
            abs_resid_tr = np.abs(y_tr - p_tr_ens)
        k_idx = min(int(np.ceil(0.80 * len(abs_resid_tr))) - 1, len(abs_resid_tr) - 1)
        conf_margins[t_yr] = float(np.sort(abs_resid_tr)[max(0, k_idx)])

        r2_all      = r2_score(y_te, ens_gated) if len(y_te) > 1 and np.var(y_te) > 1e-16 else float('nan')
        rmse_f      = float(np.sqrt(mean_squared_error(y_te, ens_gated)))
        suppressed  = int((fire_prob < HURDLE_GATE_FOLD).sum())
        fire_recall = float(np.sum((ens_gated > 0) & (y_te > 0)) / max(np.sum(y_te > 0), 1))
        print(f"  Fold {t_yr} (fire-season n={te_mask.sum()}) | R2={r2_all:.4f} | RMSE={rmse_f:.4e} | "
              f"Gate={HURDLE_GATE_FOLD:.2f} (sup={suppressed}) | Recall={fire_recall:.3f} | "
              f"Weights=[{w_ens[0]:.2f},{w_ens[1]:.2f},{w_ens[2]:.2f}] | Iso={calib_status}")

    HURDLE_GATE = float(np.median(_fold_gates)) if _fold_gates else 0.30
    best_f2     = float(np.mean(_fold_f2s)) if _fold_f2s else 0.0
    region_hurdle_gates[reg_name] = HURDLE_GATE
    region_hurdle_gate[reg_name]  = float(HURDLE_GATE)

    region_hyperparams[reg_name] = {
        'cb_best_params': dict(cb_study.best_params),
        'cb_best_cv_r2': float(cb_study.best_value),
        'cb_best_cv_rmse': float(cb_study.best_value),
        'cb_n_trials': len(cb_study.trials),
        'xgb_best_params': dict(xgb_study.best_params),
        'xgb_best_cv_r2': float(xgb_study.best_value),
        'xgb_best_cv_rmse': float(xgb_study.best_value),
        'xgb_n_trials': len(xgb_study.trials),
        'et_params': dict(ET_PARAMS),
        'hurdle_gate_threshold': float(HURDLE_GATE),
        'hurdle_gate_f2': float(best_f2),
        'region_quantile_alpha': float(REGION_QUANTILE_ALPHA),
    }
    TUNED_HYPERPARAMS[reg_name] = {
        'CatBoost': {
            'optuna_best_params': {k: _json_safe(v) for k, v in cb_study.best_params.items()},
            'optuna_best_value_r2': float(cb_study.best_value),
            'optuna_best_value_mse': float(cb_study.best_value),
            'n_optuna_trials': int(nt),
            'final_params_used': {k: _json_safe(v) for k, v in CB_PARAMS.items()},
        },
        'XGBoost': {
            'optuna_best_params': {k: _json_safe(v) for k, v in xgb_study.best_params.items()},
            'optuna_best_value_r2': float(xgb_study.best_value),
            'optuna_best_value_mse': float(xgb_study.best_value),
            'n_optuna_trials': int(nt),
            'final_params_used': {k: _json_safe(v) for k, v in XGB_PARAMS.items()},
        },
        'ExtraTrees_fixed_not_tuned': {k: _json_safe(v) for k, v in ET_PARAMS.items()},
    }

    conformal_margins[reg_name]       = conf_margins
    non_fire_season_fpr[reg_name]     = {
        'false_positives': int(_nonfire_fp_cnt),
        'total_nonfire_negatives': int(_nonfire_tot_cnt),
        'fpr': float(_nonfire_fp_cnt / max(_nonfire_tot_cnt, 1)),
    }
    y_pred_final_ba[global_rows]      = y_pred_reg
    y_pred_ba_prehurdle[global_rows]  = y_pred_prehurdle_reg
    y_pred_raw_ens[global_rows]       = y_pred_raw_reg
    y_pred_ba_ridge[global_rows]      = y_pred_reg
    y_pred_ba_cb[global_rows]         = y_cb_reg
    y_pred_ba_xgb[global_rows]        = y_xgb_reg
    y_pred_ba_et[global_rows]         = y_et_reg
    fire_prob_all[global_rows]        = fire_prob_reg
    y_pred_hurdle_prob[global_rows]   = fire_prob_reg
    y_fire_prob[global_rows]          = fire_prob_reg
    y_true_fire_bin[global_rows]      = y_bin_reg
    y_pred_fire_bin[global_rows]      = pred_fire_bin_reg
    y_used_hurdle[global_rows]        = used_hurdle_reg
    y_hurdle_zeroed[global_rows]      = hurdle_zeroed_reg
    y_train_max_ba[global_rows]       = train_max_reg
    y_raw_ens_max_ba[global_rows]     = raw_ens_max_reg

    if imp_weight > 0:
        region_feature_importance[reg_name]  = (imp_accum_f / imp_weight, list(reg_feats))
        REGION_FEATURE_IMPORTANCE[reg_name]  = (imp_accum_f / imp_weight, list(reg_feats))
        all_region_feat_importance[reg_name] = (imp_accum_f / imp_weight, list(reg_feats))

    valid_g = (cal_yr_reg >= TEST_YEAR_START) & (~np.isnan(y_pred_reg))
    if valid_g.sum() > 1:
        print(f"[{reg_name}] FIRE-SEASON OVERALL R2: {r2_score(y_reg[valid_g], y_pred_reg[valid_g]):.4f} | "
              f"RMSE: {np.sqrt(mean_squared_error(y_reg[valid_g], y_pred_reg[valid_g])):.4e} | "
              f"Non-fire-season FPR: {non_fire_season_fpr[reg_name]['fpr']:.4f}")

for yr in test_years:
    ms, mn = 0.0, 0
    for r in region_defs.keys():
        if r in conformal_margins and yr in conformal_margins[r]:
            ms += conformal_margins[r][yr]; mn += 1
    conformal_margin_by_fold_domain[yr] = ms / max(1, mn)

region_hurdle_gates['Global'] = float(np.mean(list(region_hurdle_gates.values()))) if region_hurdle_gates else 0.30

# ==============================================================================
# FOUR BENCHMARK BASELINES ON IDENTICAL FIRE-SEASON TEST WINDOW (Sec 2.3.6)
#   (i)   Pixel x calendar-month climatology (2003-2011 training period)
#   (ii)  Persistence (previous month observed BA, y_{p, t-1})
#   (iii) Previous-year same-month observed BA (y_{p, t-12})
#   (iv)  Combined OLS linear baseline of (i)-(iii) fitted on 2003-2011
# ==============================================================================
_tr_global_mask = (YEAR_START + (time_idn // 12)) < TEST_YEAR_START
_mo_all = (time_idn % 12).astype(np.int32)
y_base_clim = np.zeros(n_rows, dtype=np.float32)
for _pid in range(n_idn_pixels):
    _pm = pixel_inv == _pid
    _pm_tr = _pm & _tr_global_mask
    for _m in range(12):
        _m_tr = _pm_tr & (_mo_all == _m)
        _cval = float(np.nanmean(y_ba_idn[_m_tr])) if _m_tr.sum() > 0 else 0.0
        y_base_clim[_pm & (_mo_all == _m)] = _cval

y_base_pers = y_lag1_global.copy().astype(np.float32)
y_base_prev = y_lag12_global.copy().astype(np.float32)

_X_ols_all = np.column_stack([np.ones(n_rows, dtype=np.float64), y_base_clim, y_base_pers, y_base_prev])
_ols_fit_m = _tr_global_mask & (time_idn >= 12)
try:
    _ols_coef, *_ = np.linalg.lstsq(_X_ols_all[_ols_fit_m], y_ba_idn[_ols_fit_m].astype(np.float64), rcond=None)
except Exception:
    _ols_coef = np.array([0.0, 1.0, 0.0, 0.0])
y_base_ols = np.maximum(0.0, (_X_ols_all @ _ols_coef)).astype(np.float32)

def _eval_baseline_dict(mask_eval):
    if mask_eval.sum() < 2:
        return {}
    yt = y_ba_idn[mask_eval]
    yp = y_pred_final_ba[mask_eval]
    model_rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    model_rho  = float(spearmanr(yt, yp)[0]) if np.std(yp) > 0 and np.std(yt) > 0 else 0.0
    out = {
        'model': {
            'rmse': model_rmse,
            'r2': float(r2_score(yt, yp)),
            'mae': float(mean_absolute_error(yt, yp)),
            'spearman_r': model_rho,
            'n': int(mask_eval.sum()),
        }
    }
    for bname, bpred in [
        ('climatology',  y_base_clim[mask_eval]),
        ('persistence',  y_base_pers[mask_eval]),
        ('previous_year', y_base_prev[mask_eval]),
        ('ols_combined', y_base_ols[mask_eval]),
    ]:
        b_rmse = float(np.sqrt(mean_squared_error(yt, bpred)))
        b_rho  = float(spearmanr(yt, bpred)[0]) if np.std(bpred) > 0 and np.std(yt) > 0 else 0.0
        out[bname] = {
            'rmse': b_rmse,
            'r2': float(r2_score(yt, bpred)),
            'mae': float(mean_absolute_error(yt, bpred)),
            'spearman_r': b_rho,
            'skill_score_vs_baseline': float(1.0 - model_rmse / max(b_rmse, 1e-12)),
        }
    return out

valid_all = ((YEAR_START + (time_idn // 12)) >= TEST_YEAR_START) & (~np.isnan(y_pred_final_ba))
baseline_evaluation = {'Domain': _eval_baseline_dict(valid_all), 'Per_Subregion': {}}
for _rn, _rdef in region_defs.items():
    _rm = _rdef[0] if isinstance(_rdef, tuple) else _rdef
    baseline_evaluation['Per_Subregion'][_rn] = _eval_baseline_dict(valid_all & _rm)

print(f"\n{'='*80}")
print(" BENCHMARK BASELINE COMPARISON (FIRE-SEASON TEST FOLDS 2012-2020)")
print(f"{'='*80}")
if 'model' in baseline_evaluation['Domain']:
    _dm = baseline_evaluation['Domain']
    print(f"  Model         : RMSE={_dm['model']['rmse']:.4e} | R2={_dm['model']['r2']:.4f} | Spearman r={_dm['model']['spearman_r']:.3f}")
    for _bn in ('climatology', 'persistence', 'previous_year', 'ols_combined'):
        _b = _dm[_bn]
        print(f"  {_bn:<14s}: RMSE={_b['rmse']:.4e} | Spearman r={_b['spearman_r']:.3f} | Skill Score={_b['skill_score_vs_baseline']:+.3f}")

# Backward-compatibility variables for downstream evaluation/export cells
test_mask0     = (time_idn >= (TEST_YEAR_START - YEAR_START) * 12)
valid_meta     = ~np.isnan(y_pred_final_ba)
calib_yrs      = np.array([YEAR_START + (t // 12) for t in time_idn])
pixel_mean_ba  = np.full(n_idn_pixels, np.nan, dtype=np.float32)
_calib_mask_pm = valid_meta & test_mask0
for _pid in range(n_idn_pixels):
    _pm = pixel_inv == _pid; _te = _pm & _calib_mask_pm
    if _te.sum() >= 1:
        pixel_mean_ba[_pid] = float(np.nanmean(y_ba_idn[_te]))
row_pm_ba = pixel_mean_ba[pixel_inv]
_iv = pixel_mean_ba[~np.isnan(pixel_mean_ba)]
if len(_iv) > 0:
    _iq = np.quantile(_iv, [0.0, .2, .4, .6, .8])
    iso_tiers = [('Q1', _iq[0], _iq[1]), ('Q2', _iq[1], _iq[2]), ('Q3', _iq[2], _iq[3]),
                 ('Q4', _iq[3], _iq[4]), ('Q5', _iq[4], 1e9)]
else:
    iso_tiers = [('Q1', 0, 1e9)]
pix_lats = unique_pixels[:, 0]; pix_lons = unique_pixels[:, 1]
feat_labels = ['Two-stage pre-test pruned regional features']

### EXPORT — Optuna-Tuned Hyperparameters (Per-Region, Final Values Used)

Captures, per pooled region, the hyperparameters selected by the per-region
Optuna search in the training cell above for CatBoost and XGBoost -- the
only two models in this ensemble that are actually Optuna-tuned. ExtraTrees
uses a fixed, untuned configuration and is included below purely for
reference/completeness (clearly flagged as not Optuna-tuned), so this one
export has the complete, final hyperparameter picture for every model in
the region's ensemble. Writes:
1. A long-format CSV, one row per (region, model, hyperparameter): the
Optuna-searched value where applicable, the fixed/non-tuned constants
merged in afterward to build the model actually trained (iterations,
loss_function, task_type, etc.), the Optuna search budget (n trials),
and the best cross-validated objective value Optuna found -- ready to
drop straight into a manuscript supplementary table.
2. A pickle of the raw nested TUNED_HYPERPARAMS dict (region -> model ->
{optuna_best_params, optuna_best_value_mse, n_optuna_trials,
final_params_used}) for anyone who wants the exact structure back
programmatically, e.g. to re-instantiate a model with the tuned
params without re-running Optuna.


In [ ]:
import os
import pickle
import pandas as pd

if 'TUNED_HYPERPARAMS' not in dir() or not TUNED_HYPERPARAMS:
    print("No TUNED_HYPERPARAMS available -- run the region training loop above first.")
else:
    _export_dir = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'

    _rows = []
    for _reg_name, _models in TUNED_HYPERPARAMS.items():
        for _model_name, _info in _models.items():
            if _model_name == 'ExtraTrees_fixed_not_tuned':
                for _pname, _pval in _info.items():
                    _rows.append(dict(region=_reg_name, model='ExtraTrees',
                                       optuna_tuned=False, param=_pname,
                                       value=_pval, source='fixed_constant'))
                continue
            _tuned_names = set(_info['optuna_best_params'].keys())
            for _pname, _pval in _info['optuna_best_params'].items():
                _rows.append(dict(region=_reg_name, model=_model_name,
                                   optuna_tuned=True, param=_pname,
                                   value=_pval, source='optuna_search'))
            for _pname, _pval in _info['final_params_used'].items():
                if _pname in _tuned_names:
                    continue  # already listed above as the Optuna-tuned value
                _rows.append(dict(region=_reg_name, model=_model_name,
                                   optuna_tuned=False, param=_pname,
                                   value=_pval, source='fixed_constant'))
            _rows.append(dict(region=_reg_name, model=_model_name,
                               optuna_tuned=None, param='n_optuna_trials',
                               value=_info['n_optuna_trials'], source='search_budget'))
            _rows.append(dict(region=_reg_name, model=_model_name,
                               optuna_tuned=None, param='optuna_best_value_mse',
                               value=_info['optuna_best_value_mse'], source='search_result'))

    tuned_hyperparams_df = pd.DataFrame(_rows)

    _csv_path = os.path.join(_export_dir, 'bbur_forecaster_maritime_west_tuned_hyperparameters.csv')
    _pkl_path = os.path.join(_export_dir, 'bbur_forecaster_maritime_west_tuned_hyperparameters.pkl')

    tuned_hyperparams_df.to_csv(_csv_path, index=False)
    with open(_pkl_path, 'wb') as f:
        pickle.dump({
            'active_region': ACTIVE_REGION,
            'tuned_hyperparams_raw': TUNED_HYPERPARAMS,
            'tuned_hyperparams_df': tuned_hyperparams_df,
        }, f)

    print(f"[Hyperparameter export] CSV -> {_csv_path} ({len(tuned_hyperparams_df)} rows)")
    print(f"[Hyperparameter export] Pickle -> {_pkl_path}")

Map the model outputs to the variable names used by the downstream export cells.


In [ ]:
import numpy as np

if 'y_pred_final_ba' in dir():
    y_pred_ba_cb  = y_pred_final_ba.copy()
    y_pred_ba_xgb = y_pred_final_ba.copy()
    y_pred_ba_et  = y_pred_final_ba.copy()

# FIX: y_pred_hurdle_prob is already built by the pooled training loop
# (Cell 22), filled per-region with real classifier probabilities. The
# old line here referenced 'fire_prob_all', a variable from the prior
# architecture that no longer exists -- since it was never defined, this
# unconditionally fell to the else-branch and silently overwrote the
# correct array with all-NaN, which is why every hurdle-derived output
# downstream (spatial hurdle maps, isolated hurdle metrics, the Cell 50
# assert) saw zero valid predictions. Only fall back to NaNs if the
# pooled loop's array is genuinely missing.
if 'y_pred_hurdle_prob' not in dir():
    y_pred_hurdle_prob = np.full(len(y_pred_final_ba) if 'y_pred_final_ba' in dir() else 0, np.nan, dtype=np.float32)

# FIX: region_hurdle_gate values are plain floats (Cell 22:
# region_hurdle_gate[reg_name] = HURDLE_GATE), not {'threshold': ...}
# dicts, so the old v['threshold'] lookup would TypeError the moment
# anything actually used regional_hurdle_gates.
if 'region_hurdle_gate' in dir():
    regional_hurdle_gates = dict(region_hurdle_gate)
else:
    regional_hurdle_gates = {}

regional_hurdle_eval = {}

if 'region_feature_importance' in dir():
    all_region_feat_importance = {
        k: {'names': v[1], 'importance': v[0]}
        for k, v in region_feature_importance.items()
    }
else:
    all_region_feat_importance = {}

feat_labels = feature_names_ext if 'feature_names_ext' in dir() else []

if 'unique_pixels' in dir() and hasattr(unique_pixels, 'shape') and len(unique_pixels.shape) == 2 and unique_pixels.shape[1] >= 2:
    pix_lats = unique_pixels[:, 0]
    pix_lons = unique_pixels[:, 1]
else:
    pix_lats = np.array([])
    pix_lons = np.array([])

Compute global and regional evaluation metrics for the export.


In [ ]:
import scipy.stats
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
test_mask0 = time_idn >= (test_years[0] - YEAR_START) * 12
test_eval = test_mask0 & ~np.isnan(y_pred_final_ba)
EVAL_YEAR_START = test_years[0]
EVAL_YEAR_END = test_years[-1]
yt = y_ba_idn[test_eval]
yp = y_pred_final_ba[test_eval]
_yt_var_ok = len(yt) > 1 and np.var(yt) > 1e-12
r2_ba = r2_score(yt, yp) if _yt_var_ok else np.nan
rmse_ba = np.sqrt(mean_squared_error(yt, yp)) if len(yt) > 1 else np.nan
bias_ba = float(np.mean(yp - yt)) if len(yt) > 1 else np.nan
pearson = float(np.corrcoef(yt, yp)[0, 1]) if len(yt) > 1 else np.nan
spearman_ba, spearman_pvalue_ba = scipy.stats.spearmanr(yt, yp) if len(yt) > 1 else (np.nan, np.nan)
spearman_ba = float(spearman_ba)
spearman_pvalue_ba = float(spearman_pvalue_ba)
is_kalimantan_peat_px = np.zeros(len(unique_pixels), dtype=bool)
is_kalimantan_peat_px[pixel_inv[is_kalimantan_peat.astype(bool)]] = True
is_nonpeat_monsoonal_px = np.zeros(len(unique_pixels), dtype=bool)
is_nonpeat_monsoonal_px[pixel_inv[is_nonpeat_monsoonal.astype(bool)]] = True
is_nonpeat_savanna_px = np.zeros(len(unique_pixels), dtype=bool)
is_nonpeat_savanna_px[pixel_inv[is_nonpeat_savanna.astype(bool)]] = True
is_nonpeat_combined_px = is_nonpeat_monsoonal_px | is_nonpeat_savanna_px
_region_defs_eval = [('Peat', is_kalimantan_peat_px), ('NonPeat', is_nonpeat_combined_px), ('  NonPeat_Monsoonal', is_nonpeat_monsoonal_px), ('  NonPeat_Savanna', is_nonpeat_savanna_px)]
_region_defs = [('Peat', is_kalimantan_peat_px), ('NonPeat_Monsoonal', is_nonpeat_monsoonal_px), ('NonPeat_Savanna', is_nonpeat_savanna_px)]
_hurdle_gate_cache = region_hurdle_gate
for reg_name, reg_flag in _region_defs_eval:
    reg_pixels = np.where(reg_flag)[0]
    if len(reg_pixels) == 0:
        continue
    reg_mask = np.isin(pixel_inv, reg_pixels)
    reg_eval_mask = test_eval & reg_mask
    ytr = y_ba_idn[reg_eval_mask]
    ypr = y_pred_final_ba[reg_eval_mask]
    if len(ytr) > 1 and np.var(ytr) > 1e-12:
        reg_r2 = r2_score(ytr, ypr)
        reg_rmse = np.sqrt(mean_squared_error(ytr, ypr))

### Comprehensive Final Save


In [ ]:
import pickle, os
import numpy as np
import scipy.stats
from sklearn.metrics import (r2_score, mean_squared_error, precision_score, recall_score,
                              f1_score, roc_auc_score, average_precision_score, confusion_matrix)

# ── Per-pixel region label + hurdle-gate lookup ──────────────────────────────
# Recomputed directly from pixel centroids (pix_lats/pix_lons), since the is_*
# region flags built during feature engineering (Cell 5) are indexed by ROW,
# not by pixel id, and can't be indexed by `pid` directly. Geography and
# thresholds mirror Cell 5/Cell 8 exactly.
_core_box_pix    = (pix_lats >= -6.5) & (pix_lats <= 7.0) & (pix_lons >= 94.5) & (pix_lons <= 119.6)
_java_bali_pix   = ((pix_lats >= -9.0) & (pix_lats <= -5.7) & (pix_lons >= 105.0) & (pix_lons <= 114.7)) | ((pix_lats >= -9.0) & (pix_lats <= -7.9) & (pix_lons >= 114.3) & (pix_lons <= 115.9))
_sulawesi_pix    = (pix_lats >= -6.5) & (pix_lats <= 1.9)  & (pix_lons >= 118.5) & (pix_lons <= 125.5)
_nusa_tenggara_pix = (pix_lats >= -11.2) & (pix_lats <= -6.8) & (pix_lons >= 114.8) & (pix_lons <= 127.5) & (~_java_bali_pix)
_maluku_pix         = (pix_lats >= -8.8)  & (pix_lats <= 3.2)  & (pix_lons >= 123.8) & (pix_lons <= 135.2) & (~_sulawesi_pix) & (~_nusa_tenggara_pix)
_nusa_tenggara_maluku_pix = _nusa_tenggara_pix | _maluku_pix
_pm_pix          = (pix_lats >= 1.0) & (pix_lats <= 7.0) & (pix_lons >= 100.0) & (pix_lons <= 104.5)
_phl_pix         = (pix_lats >= 5.0) & (pix_lats <= 21.5) & (pix_lons >= 116.5) & (pix_lons <= 127.5) & ~((pix_lats <= 7.5) & (pix_lons >= 116.0) & (pix_lons <= 119.6))
_core_minus_java_pix = _core_box_pix & (~_java_bali_pix) & (~_sulawesi_pix) & (~_pm_pix) & (~_phl_pix)

is_peat_pix = np.zeros(n_idn_pixels, dtype=bool)
for _pid in range(n_idn_pixels):
    _pm = pixel_inv == _pid
    if _pm.sum() > 0:
        is_peat_pix[_pid] = bool(np.mean(is_peat_idn[_pm]) >= 0.5)

is_kalimantan_peat_pix            = _core_minus_java_pix & is_peat_pix
is_sumatra_kalimantan_nonpeat_pix = _core_minus_java_pix & (~is_peat_pix)
is_nonpeat_monsoonal_pix = is_sumatra_kalimantan_nonpeat_pix | _java_bali_pix
is_nonpeat_savanna_pix   = _sulawesi_pix | _nusa_tenggara_maluku_pix

region_label_pix = np.full(n_idn_pixels, 'Other', dtype=object)
region_label_pix[is_kalimantan_peat_pix]   = 'Peat'
region_label_pix[is_nonpeat_monsoonal_pix] = 'NonPeat_Monsoonal'
region_label_pix[is_nonpeat_savanna_pix]   = 'NonPeat_Savanna'

_gate_lookup = dict(_hurdle_gate_cache) if '_hurdle_gate_cache' in dir() else {}
gate_pix = np.array([float(_gate_lookup.get(r, 0.20)) for r in region_label_pix], dtype=np.float32)

# ── Per-pixel grids + flat arrays: obs/pred/bias/R2/p-value/RMSE + hurdle P/R/F1 ──
lat_u = np.unique(pix_lats)
lon_u = np.unique(pix_lons)
lat_idx = {l: i for i, l in enumerate(lat_u)}
lon_idx = {l: i for i, l in enumerate(lon_u)}
H, W = len(lat_u), len(lon_u)

obs_grid   = np.full((H, W), np.nan)
prd_grid   = np.full((H, W), np.nan)
bias_grid  = np.full((H, W), np.nan)
r2_grid    = np.full((H, W), np.nan)
_MIN_ROWS_FOR_R2_METRIC = 4   # need >= 4 valid test rows
_MIN_FIRE_FOR_R2_METRIC = 6   # need >= 6 actual fire months -- bumped from 2
                              # (was declared but never referenced by the loop
                              # below, which hardcoded 2 either way -- R2/p-value
                              # at n=2-3 is close to meaningless, hence the
                              # speckled/checkerboard spatial R2 and p-value maps)
pval_grid  = np.full((H, W), np.nan)
spearman_grid      = np.full((H, W), np.nan)   # Spearman rank correlation (obs vs pred), per pixel
spearman_pval_grid = np.full((H, W), np.nan)
rmse_grid              = np.full((H, W), np.nan)
hurdle_precision_grid  = np.full((H, W), np.nan)
hurdle_recall_grid     = np.full((H, W), np.nan)
hurdle_f1_grid         = np.full((H, W), np.nan)
hurdle_n_grid          = np.full((H, W), np.nan)

pix_n_obs      = np.zeros(n_idn_pixels, dtype=np.int32)
pix_obs_mean   = np.full(n_idn_pixels, np.nan, dtype=np.float32)
pix_pred_mean  = np.full(n_idn_pixels, np.nan, dtype=np.float32)
pix_bias       = np.full(n_idn_pixels, np.nan, dtype=np.float32)
pix_r2         = np.full(n_idn_pixels, np.nan, dtype=np.float32)
pix_pval       = np.full(n_idn_pixels, np.nan, dtype=np.float32)
pix_spearman      = np.full(n_idn_pixels, np.nan, dtype=np.float32)
pix_spearman_pval = np.full(n_idn_pixels, np.nan, dtype=np.float32)
pix_rmse       = np.full(n_idn_pixels, np.nan, dtype=np.float32)
pix_hurdle_n         = np.zeros(n_idn_pixels, dtype=np.int32)
pix_hurdle_precision = np.full(n_idn_pixels, np.nan, dtype=np.float32)
pix_hurdle_recall    = np.full(n_idn_pixels, np.nan, dtype=np.float32)
pix_hurdle_f1        = np.full(n_idn_pixels, np.nan, dtype=np.float32)

for pid in range(n_idn_pixels):
    pm = (pixel_inv == pid)

    yt_all = y_ba_idn[pm]; yp_all = y_pred_final_ba[pm]
    valid = ~np.isnan(yp_all)
    m_obs = np.nan; m_prd = np.nan
    if valid.sum() >= _MIN_ROWS_FOR_R2_METRIC:
        yt, yp = yt_all[valid], yp_all[valid]
        m_obs = np.mean(yt); m_prd = np.mean(yp)
        pix_n_obs[pid]     = int(valid.sum())
        pix_obs_mean[pid]  = m_obs
        pix_pred_mean[pid] = m_prd
        pix_bias[pid]      = m_prd - m_obs
        pix_rmse[pid]      = np.sqrt(mean_squared_error(yt, yp))
        if int((yt > 0).sum()) < _MIN_FIRE_FOR_R2_METRIC: continue  # skip pixels below the fire-month floor
        _r2_raw = r2_score(yt, yp)
        pix_r2[pid] = _r2_raw if _r2_raw >= -1.0 else float('nan')
        if np.std(yt) > 0 and np.std(yp) > 0:
            _, p = scipy.stats.pearsonr(yt, yp)
            pix_pval[pid] = p
            _rho, _p_rho = scipy.stats.spearmanr(yt, yp)
            pix_spearman[pid]      = _rho
            pix_spearman_pval[pid] = _p_rho

    hurdle_all = y_pred_hurdle_prob[pm]
    valid_h = ~np.isnan(hurdle_all)
    if valid_h.sum() >= 1:
        yb_true = (yt_all[valid_h] > 0).astype(int)
        yb_prob = hurdle_all[valid_h]
        yb_pred = (yb_prob >= gate_pix[pid]).astype(int)
        pix_hurdle_n[pid]         = int(valid_h.sum())
        pix_hurdle_precision[pid] = precision_score(yb_true, yb_pred, zero_division=0)
        pix_hurdle_recall[pid]    = recall_score(yb_true, yb_pred, zero_division=0)
        pix_hurdle_f1[pid]        = f1_score(yb_true, yb_pred, zero_division=0)

    r = lat_idx[pix_lats[pid]]; c = lon_idx[pix_lons[pid]]
    obs_grid[r, c]  = m_obs
    prd_grid[r, c]  = m_prd
    bias_grid[r, c] = pix_bias[pid]
    r2_grid[r, c]   = pix_r2[pid]
    pval_grid[r, c] = pix_pval[pid]
    spearman_grid[r, c]      = pix_spearman[pid]
    spearman_pval_grid[r, c] = pix_spearman_pval[pid]
    rmse_grid[r, c] = pix_rmse[pid]
    hurdle_precision_grid[r, c] = pix_hurdle_precision[pid]
    hurdle_recall_grid[r, c]    = pix_hurdle_recall[pid]
    hurdle_f1_grid[r, c]        = pix_hurdle_f1[pid]
    hurdle_n_grid[r, c]         = pix_hurdle_n[pid]

print('Pixel-level grids/metrics computed (obs/pred/bias/R2/p-value/RMSE + hurdle precision/recall/F1).')

# --- Domain-total (this notebook's region only) monthly time series + 80% conformal margin ---
_dom_months = np.unique(time_idn[test_eval])
_dom_obs, _dom_prd = [], []
for _tm in _dom_months:
    _pm = (time_idn == _tm) & test_eval
    _dom_obs.append(float(np.nansum(y_ba_idn[_pm])))
    _dom_prd.append(float(np.nansum(y_pred_final_ba[_pm])))
_dom_obs = np.array(_dom_obs); _dom_prd = np.array(_dom_prd)
_dom_dates = [np.datetime64('1997-01') + np.timedelta64(int(m), 'M') for m in _dom_months]
# Leak-free Jackknife+ margin (Sec 2.3.5): training-cell margins (OOF residuals, cal_yr < t_yr) scaled to monthly totals
_dom_fold_m = [v for v in conformal_margin_by_fold_domain.values() if np.isfinite(v)]
_dom_margin = (float(np.mean(_dom_fold_m) * np.sqrt(max(test_eval.sum() / max(len(_dom_months), 1), 1.0)))
               if _dom_fold_m else float('nan'))

# --- Per-region monthly time series + conformal margins (Peat / NonPeat / sub-zones) ---
region_time_series = {}
for _reg_name, _reg_flag in _region_defs_eval:
    _reg_name_c = _reg_name.strip()
    _reg_pixels = np.where(_reg_flag)[0]
    if len(_reg_pixels) == 0:
        continue
    _mask = np.isin(pixel_inv, _reg_pixels) & test_mask0 & ~np.isnan(y_pred_final_ba)
    if _mask.sum() == 0:
        continue
    _yt = y_ba_idn[_mask]; _yp = y_pred_final_ba[_mask]; _tm = time_idn[_mask]
    _months = np.unique(_tm)
    _m_obs = np.array([float(_yt[_tm == m].sum()) for m in _months])
    _m_prd = np.array([float(_yp[_tm == m].sum()) for m in _months])
    _reg_fold_m = [v for v in conformal_margins.get(_reg_name, conformal_margins.get(_reg_name_c, {})).values() if np.isfinite(v)]
    _margin = (float(np.mean(_reg_fold_m) * np.sqrt(max(_mask.sum() / max(len(_months), 1), 1.0)))
               if _reg_fold_m else float('nan'))
    _dates = [np.datetime64('1997-01') + np.timedelta64(int(m), 'M') for m in _months]
    _r2v = r2_score(_m_obs, _m_prd) if len(_m_obs) > 1 and np.var(_m_obs) > 1e-12 else float('nan')
    region_time_series[_reg_name_c] = {
        'months': _months, 'dates': _dates, 'obs': _m_obs, 'pred': _m_prd,
        'conformal_margin_80': _margin, 'monthly_total_r2': _r2v,
    }

# --- Full hurdle-classifier summary at the deployed per-region gate ---
_test_eval_h = test_mask0 & ~np.isnan(y_pred_hurdle_prob)
_y_true_h = (y_ba_idn[_test_eval_h] > 0).astype(int)
_y_prob_h = y_pred_hurdle_prob[_test_eval_h]
_row_gate_h = gate_pix[pixel_inv[_test_eval_h]]
_y_pred_h = (_y_prob_h >= _row_gate_h).astype(int)
hurdle_summary = {
    'roc_auc':   float(roc_auc_score(_y_true_h, _y_prob_h)) if len(np.unique(_y_true_h)) > 1 else float('nan'),
    'pr_auc':    float(average_precision_score(_y_true_h, _y_prob_h)) if len(np.unique(_y_true_h)) > 1 else float('nan'),
    'precision': float(precision_score(_y_true_h, _y_pred_h, zero_division=0)),
    'recall':    float(recall_score(_y_true_h, _y_pred_h, zero_division=0)),
    'f1':        float(f1_score(_y_true_h, _y_pred_h, zero_division=0)),
    'confusion_matrix': confusion_matrix(_y_true_h, _y_pred_h).tolist(),
    'base_rate': float(_y_true_h.mean()),
    'gate_by_region': dict(_hurdle_gate_cache),
}

# --- Per-region aggregate R2/RMSE table ---
regional_metrics = {}
for _reg_name, _reg_flag in _region_defs_eval:
    _reg_name_c = _reg_name.strip()
    _reg_pixels = np.where(_reg_flag)[0]
    if len(_reg_pixels) == 0:
        continue
    _reg_mask = np.isin(pixel_inv, _reg_pixels)
    _reg_eval_mask = test_eval & _reg_mask
    _ytr = y_ba_idn[_reg_eval_mask]; _ypr = y_pred_final_ba[_reg_eval_mask]
    if len(_ytr) > 1 and np.var(_ytr) > 1e-12:
        regional_metrics[_reg_name_c] = {
            'n': int(len(_ytr)), 'r2': float(r2_score(_ytr, _ypr)),
            'rmse': float(np.sqrt(mean_squared_error(_ytr, _ypr))),
        }
    else:
        regional_metrics[_reg_name_c] = {'n': int(len(_ytr)), 'r2': float('nan'), 'rmse': float('nan')}

# --- Flat pixel-level table: the primary artifact for stitching a full SE Asia map ---
pixel_level = {
    'lat': pix_lats.astype(np.float32),
    'lon': pix_lons.astype(np.float32),
    'region_label': region_label_pix,
    'is_peat': is_peat_pix,
    'n_obs': pix_n_obs,
    'obs_mean_ba': pix_obs_mean,
    'pred_mean_ba': pix_pred_mean,
    'bias': pix_bias,
    'r2': pix_r2,
    'pearson_pvalue': pix_pval,
    'spearman_r': pix_spearman,
    'spearman_pvalue': pix_spearman_pval,
    'rmse': pix_rmse,
    'hurdle_gate': gate_pix,
    'hurdle_n': pix_hurdle_n,
    'hurdle_precision': pix_hurdle_precision,
    'hurdle_recall': pix_hurdle_recall,
    'hurdle_f1': pix_hurdle_f1,
}

# --- Gridded arrays (this notebook's own bounding box -- quick standalone replot) ---
spatial_grids = {
    'lat_u': lat_u, 'lon_u': lon_u,
    'obs_grid': obs_grid, 'prd_grid': prd_grid, 'bias_grid': bias_grid,
    'r2_grid': r2_grid, 'pval_grid': pval_grid,
    'spearman_grid': spearman_grid, 'spearman_pval_grid': spearman_pval_grid,
    'rmse_grid': rmse_grid,
    'hurdle_precision_grid': hurdle_precision_grid,
    'hurdle_recall_grid': hurdle_recall_grid,
    'hurdle_f1_grid': hurdle_f1_grid,
    'hurdle_n_grid': hurdle_n_grid,
}

comprehensive_results = {
    'baseline_evaluation':       baseline_evaluation,   # Sec 2.3.6: 4 baselines, RMSE / Spearman / skill score
    'non_fire_season_fpr':       non_fire_season_fpr,   # Sec 2.1/2.3.6: non-fire-season false-positive rate per sub-region
    'conformal_margins_by_region_fold': conformal_margins,   # leak-free Jackknife+ margins {region: {year: margin}}
    'conformal_margin_by_fold_domain': conformal_margin_by_fold_domain,
    'metadata': {
        'active_region': ACTIVE_REGION,   # e.g. 'Maritime_West'
        'notebook': 'bbur-forecaster-west-lead1',
        'lead_time_months': 1,
        'test_years': test_years,
        'test_year_range': [test_years[0], test_years[-1]],
        'year_start': YEAR_START,
        'year_end': YEAR_END,
    },
    'global_metrics': {
        'r2': float(r2_ba), 'rmse': float(rmse_ba), 'bias': float(bias_ba),
        'pearson': float(pearson),
        'spearman': float(spearman_ba), 'spearman_pvalue': float(spearman_pvalue_ba),
        'n': int(len(y_ba_idn[test_eval])),
    },
    'regional_metrics': regional_metrics,
    'hurdle_summary': hurdle_summary,
    'pixel_level': pixel_level,
    'spatial_grids': spatial_grids,
    'time_series': {
        'domain_total': {'months': _dom_months, 'dates': _dom_dates, 'obs': _dom_obs,
                          'pred': _dom_prd, 'conformal_margin_80': _dom_margin},
        'by_region': region_time_series,
    },

    # -- Raw per-observation arrays (kept for anyone who wants to recompute a
    #    grid differently downstream, e.g. a different metric or a finer
    #    regrid -- matches the raw-array export in the Maritime East / other
    #    regional notebooks so all regions expose the same flat row-level
    #    data for stitching) --
    'raw_observations': {
        'y_ba_idn':         y_ba_idn,
        'y_pred_final_ba':  y_pred_final_ba,
        'pixel_inv':        pixel_inv,
        'time_idn':         time_idn,
        'pix_lats':         pix_lats,
        'pix_lons':         pix_lons,
        'n_idn_pixels':     n_idn_pixels,
        'hurdle_prob_all':  y_pred_hurdle_prob,
    },
}

_ckpt_dir = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
FINAL_SAVE_PATH = os.path.join(_ckpt_dir, 'bbur_forecaster_maritime_west_final_comprehensive.pkl')
with open(FINAL_SAVE_PATH, 'wb') as f:
    pickle.dump(comprehensive_results, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f'Comprehensive results saved -> {FINAL_SAVE_PATH}')
print(f'  Pixels           : {n_idn_pixels}')
print(f'  Regions          : {list(regional_metrics.keys())}')
print(f'  Global R2/RMSE   : {r2_ba:.4f} / {rmse_ba:.4e}')
print(f'  Hurdle ROC-AUC   : {hurdle_summary["roc_auc"]:.4f}  PR-AUC: {hurdle_summary["pr_auc"]:.4f}')
print(f'  Raw obs rows      : {len(y_ba_idn):,}  (raw_observations[...] -- y_ba_idn/y_pred_final_ba/pixel_inv/time_idn/pix_lats/pix_lons/n_idn_pixels/hurdle_prob_all)')

Save the results checkpoint.


In [ ]:
import pickle, os
CHECKPOINT_PATH = 'bbur_forecaster_maritime_west_checkpoint.pkl'
checkpoint = {
    'y_ba_idn':        y_ba_idn,
    'y_pred_final_ba': y_pred_final_ba,
    'y_pred_ba_cb':    y_pred_ba_cb,
    'y_pred_ba_xgb':   y_pred_ba_xgb,
    'y_pred_ba_et':    y_pred_ba_et,
    'pixel_inv':       pixel_inv,
    'time_idn':        time_idn,
    'unique_pixels':   unique_pixels,
    'pix_lats':        pix_lats,
    'pix_lons':        pix_lons,
    'n_idn_pixels':    n_idn_pixels,
    'feat_labels':  feat_labels,
    'all_region_feat_importance': all_region_feat_importance,
    'test_years':      test_years,
    'TEST_YEAR_START': TEST_YEAR_START,
    'YEAR_START':      YEAR_START,
    'YEAR_END':        YEAR_END,
    'r2_ba':    r2_ba,
    'rmse_ba':  rmse_ba,
    'bias_ba':  bias_ba,
    'pearson':  pearson,
    'y_pred_hurdle_prob':   y_pred_hurdle_prob,
    # y_pred_raw_reg / y_pred_raw_tree_only dropped (ported full architecture
    # swap): Tweedie-blend leftovers not produced by the pooled architecture.
    'hurdle_gate_by_region': dict(_hurdle_gate_cache),
}
with open(CHECKPOINT_PATH, 'wb') as f:
    pickle.dump(checkpoint, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f'Saved checkpoint -> {CHECKPOINT_PATH}')

Optionally reload the saved checkpoint instead of retraining.


In [ ]:
import pickle, os
CHECKPOINT_PATH = 'bbur_forecaster_maritime_west_checkpoint.pkl'
LOAD_CHECKPOINT = os.path.exists(CHECKPOINT_PATH)
if LOAD_CHECKPOINT:
    with open(CHECKPOINT_PATH, 'rb') as f:
        _ckpt = pickle.load(f)
    globals().update(_ckpt)
    print(f'Loaded checkpoint from {CHECKPOINT_PATH}')
else:
    print('No checkpoint found - run the full training pipeline above first.')

Define the manuscript predictor categories.


In [ ]:
MANUSCRIPT_CATEGORIES = [
    'Static land-surface / fuel-load state',
    'Climate teleconnection & compound-drought indices',
    'Meteorological reanalysis & derived moisture amplifiers (ERA5)',
    'Fire-history predictors (GFED4s)',
    'Vegetation fuel-moisture state',
    'Other (spatial / calendar / region-specific)',
]

def categorize_feature(name: str) -> str:
    """Map one feature name to one of the manuscript predictor categories (Sec 2.2)."""
    f = name.lower()

    if (f.startswith('defo_ba') or f.startswith('crop_ba')
            or f in ('years_since_first_defo', 'years_since_last_fire',
                     'fire_freq_train', 'fire_freq', 'fire_frequency',
                     'defo_recent_pressure_3_6mo', 'crop_recent_pressure_1_3mo',
                     'ba_prev_year', 'defo_x_dryseason', 'defo_x_dryseason_lag1',
                     'y_lag1', 'y_lag3')
            or 'cumulative_ba' in f
            or f.startswith('burned_area_pred_lag')):
        return MANUSCRIPT_CATEGORIES[3]

    if f.startswith('ndvi') or f.startswith('evi'):
        return MANUSCRIPT_CATEGORIES[4]

    if (f in ('soc', 'soc_z', 'soc_zscore', 'soc_stock', 'static_soc',
              'soc_domain_zscore', 'soc_domain', 'peat_mask', 'peat_mask_continuous')
            or 'soil_organic_carbon' in f):
        return MANUSCRIPT_CATEGORIES[0]

    if (f.startswith('oni') or f.startswith('dmi') or f.startswith('iod')
            or f.startswith('enso')
            or f in ('oni_x_dmi', 'oni_x_dmi_lag1', 'iod_x_oni', 'iod_x_oni_lag1',
                     'png_drought_x_oni', 'png_drought_x_oni_lag1', 'png_iod_drought', 'pm_drought_x_oni', 'pm_drought_x_oni_lag1',
                     'drought_oni_soft_interact', 'drought_oni_soft_interact_lag1',
                     'is_compound_drought', 'is_compound_drought_lag1', 'strong_enso')
            or 'kbdi' in f or f.startswith('drought_stress')):
        return MANUSCRIPT_CATEGORIES[1]

    _met_markers = (
        'era5_', 'dewpoint', 'd2m', 'soil_temp', 'stl', 'swvl',
        't2m', 'u10', 'v10', 'u_850', 'v_850', 'z_850', 'w_850',
        'u_500', 'v_500', 'z_500', 'w_500', 'vpd', 'tp_', 'precip',
        'grace', 'tws', 'rainfall_deficit', 'monsoon_onset',
        'drought_breaker_flag', 'days_since_last_rain',
        'pm_coastal_x_vpd', 'pm_inland_x_swvl3deficit',
    )
    if any(m in f for m in _met_markers):
        return MANUSCRIPT_CATEGORIES[2]

    return MANUSCRIPT_CATEGORIES[5]

def build_category_map(feature_names):
    """Return {feature_name: manuscript_category} mapping."""
    return {f: categorize_feature(f) for f in feature_names}

_all_known_feats = list(feature_names_idn_full) if 'feature_names_idn_full' in dir() else (
                   list(feature_names_ext) if 'feature_names_ext' in dir() else list(feature_names))
category_map = build_category_map(_all_known_feats)
print(f'Categorized {len(category_map)} features across {len(MANUSCRIPT_CATEGORIES)} manuscript categories.')

### Grouped, Category-Normalized Permutation Importance With Fold Stability

Replaces "how many of the top-10 individually-ranked features fall in each
category" (biased by category size -- the 47-variable ERA5 block has far
more chances to place features in a top-10 list than the 2-variable
static/fuel-load block, independent of actual predictive contribution)
with grouped permutation importance (Gregorutti et al., 2015): for every
manuscript category, jointly shuffle (same random row-permutation applied
to every column in the group at once, preserving within-group correlation
structure) all of that category's features together and measure the drop
in held-out R2. This scores each category as a whole, not by counting
individual-feature rank membership, and is computed independently on every
walk-forward fold (the same test years already used for evaluation
elsewhere in this notebook) so we can report the mean AND the
fold-to-fold standard deviation -- i.e. stability/uncertainty, not just a
single point estimate. We also report the importance normalized by the
number of features in the category (average per-feature contribution),
which is the fairest apples-to-apples comparison across categories of very
different size.

The proxy model used here is a single, modest CatBoostRegressor per fold
(same hyperparameters as the pruning-proxy model already used elsewhere in
this notebook) -- NOT a retrain of the full deployed CB+XGB+ET ensemble +
isotonic + hurdle-gate pipeline. This keeps the diagnostic self-contained,
cheap (one fit per fold, reused across every category's permutations), and
decoupled from that pipeline's extra machinery, while still being scored
walk-forward on the exact same fold boundaries as the deployed model.


In [ ]:
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score

def grouped_permutation_importance_by_fold(x_full, y_full, time_full, feature_names_full,
                                            reg_feat_names, row_mask, year_start, test_years,
                                            min_train=200, min_test=5, n_repeats=5,
                                            random_state=42, cb_proxy_params=None):
    if cb_proxy_params is None:
        cb_proxy_params = dict(iterations=300, depth=5, learning_rate=0.08,
                                loss_function='RMSE', random_seed=random_state,
                                verbose=0, task_type='CPU', thread_count=4)

    used_feats = [f for f in reg_feat_names if f in feature_names_full]
    missing = [f for f in reg_feat_names if f not in feature_names_full]
    if missing:
        print(f"    [grouped-perm-importance] {len(missing)} feature(s) not found in "
              f"the feature-name list, skipped: {missing[:5]}{'...' if len(missing) > 5 else ''}")
    feat_idx = [feature_names_full.index(f) for f in used_feats]

    row_mask = np.asarray(row_mask, dtype=bool)
    X_reg = x_full[row_mask][:, feat_idx].astype(np.float32)
    y_reg = y_full[row_mask].astype(np.float32)
    t_reg = time_full[row_mask]
    cal_yr_reg = year_start + (t_reg // 12)

    cat_of = {f: categorize_feature(f) for f in used_feats}
    cats_present = sorted(set(cat_of.values()),
                           key=lambda c: MANUSCRIPT_CATEGORIES.index(c))
    cols_by_cat = {c: [i for i, f in enumerate(used_feats) if cat_of[f] == c]
                   for c in cats_present}

    rng = np.random.default_rng(random_state)
    fold_records = []  # (test_year, category, mean_delta_r2_over_repeats)
    fold_r2 = {}

    for t_yr in test_years:
        tr_mask = cal_yr_reg < t_yr
        te_mask = (cal_yr_reg == t_yr) & (~np.isnan(y_pred_final_ba[row_mask]))
        if tr_mask.sum() < min_train or te_mask.sum() < min_test:
            continue

        X_tr, y_tr = X_reg[tr_mask], y_reg[tr_mask]
        X_te, y_te = X_reg[te_mask], y_reg[te_mask]

        proxy = CatBoostRegressor(**cb_proxy_params)
        proxy.fit(X_tr, y_tr)
        base_r2 = r2_score(y_te, proxy.predict(X_te)) if len(y_te) > 1 else np.nan
        fold_r2[t_yr] = base_r2

        for cat, cols in cols_by_cat.items():
            if not cols:
                continue
            drops = []
            for _ in range(n_repeats):
                X_perm = X_te.copy()
                perm_order = rng.permutation(len(X_perm))
                X_perm[:, cols] = X_perm[perm_order][:, cols]
                perm_r2 = r2_score(y_te, proxy.predict(X_perm)) if len(y_te) > 1 else np.nan
                drops.append(base_r2 - perm_r2)
            fold_records.append((t_yr, cat, float(np.mean(drops))))

    return fold_records, cols_by_cat, fold_r2


def summarize_grouped_importance(fold_records, cols_by_cat):
    """Per-category mean importance, fold-to-fold std (stability), and the
    per-feature-normalized score, ordered to mirror MANUSCRIPT_CATEGORIES."""
    df = pd.DataFrame(fold_records, columns=['test_year', 'category', 'delta_r2'])
    rows = []
    for cat, cols in cols_by_cat.items():
        sub = df.loc[df.category == cat, 'delta_r2'].values
        if len(sub) == 0:
            continue
        rows.append(dict(
            category=cat,
            n_features=len(cols),
            n_folds=len(sub),
            mean_importance=float(np.mean(sub)),
            std_across_folds=float(np.std(sub)),
            per_feature_normalized=float(np.mean(sub)) / max(len(cols), 1),
        ))
    summary = pd.DataFrame(rows)
    if len(summary) == 0:
        return summary
    total_pos = summary['mean_importance'].clip(lower=0).sum()
    summary['pct_share'] = (summary['mean_importance'].clip(lower=0) / total_pos * 100
                             if total_pos > 0 else 0.0)
    order = {c: i for i, c in enumerate(MANUSCRIPT_CATEGORIES)}
    summary['_order'] = summary['category'].map(order)
    return summary.sort_values('_order').drop(columns='_order').reset_index(drop=True)

### MANUSCRIPT-CATEGORY FEATURE IMPORTANCE — Maritime West (Kalimantan & Sumatra)

Replaces the previous "top-N individually-ranked features" report with
grouped, category-normalized permutation importance, computed independently
on every walk-forward fold and reported as mean +/- std across folds. See
the MANUSCRIPT_CATEGORIES / grouped_permutation_importance_by_fold cells
above for the taxonomy and method. Re-derives each region's held-out rows
from `region_defs` + the region's final (post-pruning) feature list stored
in `region_feature_importance[reg_name][1]` by the training cell above, so
this cell can run standalone after that cell has finished.


In [ ]:
if 'region_feature_importance' not in dir() or not region_feature_importance:
    print("No region_feature_importance available -- run the training cell above first.")
else:
    region_grouped_summaries = {}
    region_fold_records = {}

    for reg_name, row_mask in region_defs.items():
        if reg_name not in region_feature_importance:
            print(f"[{reg_name}] No stored feature list -- skipping.")
            continue
        _reg_feat_names = list(region_feature_importance[reg_name][1])
        print(f"\n[{reg_name}] Grouped permutation importance "
              f"({len(_reg_feat_names)} features, walk-forward folds = {test_years})...")

        _fold_records, _cols_by_cat, _fold_r2 = grouped_permutation_importance_by_fold(
            x_full=x_idn, y_full=y_ba_idn, time_full=time_idn,
            feature_names_full=feature_names_idn_full,
            reg_feat_names=_reg_feat_names, row_mask=row_mask,
            year_start=YEAR_START, test_years=test_years,
        )
        region_fold_records[reg_name] = _fold_records
        _summary = summarize_grouped_importance(_fold_records, _cols_by_cat)
        region_grouped_summaries[reg_name] = _summary

        print(f"  Per-fold proxy R2: " +
              ", ".join(f"{yr}={r2:.3f}" for yr, r2 in _fold_r2.items()))
        if len(_summary) == 0:
            print("  No folds had enough data to score -- skipping this region.")
            continue
        print(f"  {'Category':<62s} {'n_feat':>6s} {'mean_dR2':>10s} {'std_folds':>10s} "
              f"{'per_feat':>10s} {'%share':>8s}")
        for _, r in _summary.iterrows():
            print(f"  {r['category']:<62s} {r['n_features']:>6d} {r['mean_importance']:>10.5f} "
                  f"{r['std_across_folds']:>10.5f} {r['per_feature_normalized']:>10.5f} "
                  f"{r['pct_share']:>7.1f}%")

### Final Top-5 Predictors (With Manuscript Category) -- After Grouped Permutation

The grouped permutation cell above scores each manuscript CATEGORY as a whole
(all of its columns shuffled together). This cell resolves that result down to
individual predictors so the "top features" can be printed with their
category: on the SAME walk-forward folds and the SAME proxy CatBoost model as
the grouped analysis, each final (post-pruning) feature is permuted on its own
and the mean drop in held-out R2 is averaged across folds. The top 5 are
printed with (i) the feature's manuscript category (categorize_feature) and
(ii) that category's grouped-permutation %share from the table above.

Caveat (printed below too): single-feature permutation under-states features
that have correlated substitutes (lags/windows of one variable); category-
level shares from the grouped cell remain the primary, size-unbiased result.


In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score

_FEATURE_NAMES_FULL = feature_names_idn_full


def top_features_by_fold_permutation(x_full, y_full, time_full, feature_names_full,
                                      reg_feat_names, row_mask, year_start, test_years,
                                      min_train=200, min_test=5, n_repeats=3,
                                      random_state=42):
    """Per-feature permutation importance (mean delta R2 over repeats), computed on
    every walk-forward fold with the same proxy model as the grouped analysis.
    Returns a DataFrame: feature, category, mean_dR2, std_across_folds, n_folds."""
    proxy_params = dict(iterations=300, depth=5, learning_rate=0.08,
                        loss_function='RMSE', random_seed=random_state,
                        verbose=0, task_type='CPU', thread_count=4)
    used = [f for f in reg_feat_names if f in feature_names_full]
    idx = [feature_names_full.index(f) for f in used]
    row_mask = np.asarray(row_mask, dtype=bool)
    X = x_full[row_mask][:, idx].astype(np.float32)
    y = y_full[row_mask].astype(np.float32)
    cal_yr = year_start + (time_full[row_mask] // 12)
    rng = np.random.default_rng(random_state)
    per_fold = {f: [] for f in used}

    for t_yr in test_years:
        tr, te = cal_yr < t_yr, (cal_yr == t_yr) & (~np.isnan(y_pred_final_ba[row_mask]))
        if tr.sum() < min_train or te.sum() < min_test:
            continue
        model = CatBoostRegressor(**proxy_params)
        model.fit(X[tr], y[tr])
        X_te, y_te = X[te], y[te]
        base = r2_score(y_te, model.predict(X_te))
        for j, f in enumerate(used):
            drops = []
            for _ in range(n_repeats):
                Xp = X_te.copy()
                Xp[:, j] = X_te[rng.permutation(len(X_te)), j]
                drops.append(base - r2_score(y_te, model.predict(Xp)))
            per_fold[f].append(float(np.mean(drops)))

    rows = [dict(feature=f, category=categorize_feature(f),
                 mean_dR2=float(np.mean(v)), std_across_folds=float(np.std(v)),
                 n_folds=len(v))
            for f, v in per_fold.items() if len(v) > 0]
    return (pd.DataFrame(rows).sort_values('mean_dR2', ascending=False)
            .reset_index(drop=True))


if 'region_feature_importance' not in dir() or not region_feature_importance:
    print("No region_feature_importance available -- run the training cell first.")
else:
    region_top5_features = {}       # reg_name -> top-5 DataFrame
    region_feature_perm_importance = {}   # reg_name -> full per-feature DataFrame
    for reg_name, row_mask in region_defs.items():
        if reg_name not in region_feature_importance:
            continue
        _feats = list(region_feature_importance[reg_name][1])
        if 'assert_no_concurrent' in dir():      # forecaster notebooks only
            assert_no_concurrent(_feats, where=f'top-5 / {reg_name}')
        _df = top_features_by_fold_permutation(
            x_full=x_idn, y_full=y_ba_idn, time_full=time_idn,
            feature_names_full=_FEATURE_NAMES_FULL,
            reg_feat_names=_feats, row_mask=row_mask,
            year_start=YEAR_START, test_years=test_years)
        region_feature_perm_importance[reg_name] = _df
        if len(_df) == 0:
            print(f"\n[{reg_name}] No folds had enough data -- no top-5 available.")
            continue
        _share = {}
        if 'region_grouped_summaries' in dir() and reg_name in region_grouped_summaries \
                and len(region_grouped_summaries[reg_name]) > 0:
            _share = dict(zip(region_grouped_summaries[reg_name]['category'],
                              region_grouped_summaries[reg_name]['pct_share']))
        _top = _df.head(5).copy()
        _top.insert(0, 'rank', range(1, len(_top) + 1))
        _top['category_grouped_%share'] = _top['category'].map(_share)
        region_top5_features[reg_name] = _top

        print(f"\n[{reg_name}] FINAL TOP-5 FEATURES after grouped permutation "
              f"({len(_feats)} final features, {int(_df['n_folds'].max())} walk-forward folds)")
        print(f"  {'#':>2s} {'Feature':<42s} {'Category':<62s} {'mean_dR2':>9s} "
              f"{'std':>8s} {'cat_%share':>10s}")
        for _, r in _top.iterrows():
            _s = r['category_grouped_%share']
            _s = f"{_s:9.1f}%" if pd.notna(_s) else f"{'n/a':>10s}"
            print(f"  {int(r['rank']):>2d} {r['feature']:<42s} {r['category']:<62s} "
                  f"{r['mean_dR2']:>9.5f} {r['std_across_folds']:>8.5f} {_s}")
    print("\nNote: per-feature permutation under-states features with correlated "
          "substitutes; category %share (grouped permutation) is the size-unbiased measure.")

### EXPORT — Grouped, Manuscript-Category Feature Importance

Dedicated export for the reviewer-requested grouped/category-normalized
permutation-importance results computed above (independent of this
notebook's other pickle exports, which run before this section and/or
still carry the old per-feature top-N dict under a different key). Writes:
1. A long-format CSV, one row per (region, manuscript category): mean
grouped permutation importance, fold-to-fold std, per-feature-
normalized score, and percent share -- ready to drop straight into a
manuscript table or supplementary spreadsheet.
2. A long-format CSV of the raw per-fold delta-R2 values behind that
summary (region, test_year, category, delta_r2), for anyone who wants
aggregation.
3. A pickle bundling both DataFrames plus the category->feature column
map, for programmatic reuse (e.g. stitching all regions/notebooks
into one cross-region manuscript table).


In [ ]:
import os
import pickle
import pandas as pd

if 'region_grouped_summaries' not in dir() or not region_grouped_summaries:
    print("No region_grouped_summaries available -- run the grouped-importance "
          "cell above first.")
else:
    _export_dir = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'

    _summary_rows = []
    for _reg_name, _summary in region_grouped_summaries.items():
        if _summary is None or len(_summary) == 0:
            continue
        _s = _summary.copy()
        _s.insert(0, 'region', _reg_name)
        _summary_rows.append(_s)
    grouped_importance_summary_df = (
        pd.concat(_summary_rows, ignore_index=True) if _summary_rows else pd.DataFrame()
    )

    _fold_rows = []
    for _reg_name, _fold_records in region_fold_records.items():
        for _test_year, _category, _delta_r2 in _fold_records:
            _fold_rows.append(dict(region=_reg_name, test_year=_test_year,
                                    category=_category, delta_r2=_delta_r2))
    grouped_importance_folds_df = pd.DataFrame(_fold_rows)

    _summary_path = os.path.join(
        _export_dir, 'bbur_forecaster_maritime_west_grouped_feature_importance_summary.csv')
    _folds_path = os.path.join(
        _export_dir, 'bbur_forecaster_maritime_west_grouped_feature_importance_folds.csv')
    _pickle_path = os.path.join(
        _export_dir, 'bbur_forecaster_maritime_west_grouped_feature_importance.pkl')

    grouped_importance_summary_df.to_csv(_summary_path, index=False)
    grouped_importance_folds_df.to_csv(_folds_path, index=False)
    with open(_pickle_path, 'wb') as f:
        pickle.dump({
            'active_region': ACTIVE_REGION,
            'manuscript_categories': MANUSCRIPT_CATEGORIES,
            'summary': grouped_importance_summary_df,
            'fold_records': grouped_importance_folds_df,
            'region_fold_records_raw': region_fold_records,
        }, f)

    print(f"[Feature-importance export] Summary CSV -> {_summary_path} "
          f"({len(grouped_importance_summary_df)} rows)")
    print(f"[Feature-importance export] Fold-level CSV -> {_folds_path} "
          f"({len(grouped_importance_folds_df)} rows)")
    print(f"[Feature-importance export] Combined pickle -> {_pickle_path}")